# 🔁 Unit 4 · Module 1 · Notebook 3: Techniques That Take More Than One Call

### Build self-consistency, prompt chaining, reflection and planning as code, and measure what each one gains and costs

**What this notebook is about.** Some well-known prompt techniques are no longer a sentence you add to a prompt, because current models reason by themselves. They still exist, in a different place: **your code makes several calls to the model and combines the results.** Asking the same question five times and keeping the most common answer is a loop and a vote. Splitting a job into three small tasks is three calls in a row. In this notebook you build four of these techniques in a few lines of code each, on real tasks from a bank's support team, and you measure what each one does and what it costs in calls, money and waiting time. Where a plain single call is a fair contest, the two stand side by side. At the end you can say, with your own numbers, which of them fits a chat where a customer is waiting and which fits a long job that runs overnight.

**The tasks we use.** A bank's support team has three jobs for a model. Each technique is tried on the job where it has a chance to help:

| Job | What the model does | How we know it is right | Technique tried on it |
|---|---|---|---|
| **15 fee calculations** | works out a total fee that needs several steps, from the bank's fee table | code compares the number with the correct one, to the cent | self-consistency, reflection |
| **20 customer complaints** | writes a reply to a message that contains three different issues | code checks that the reply states the right policy fact for each issue | prompt chaining |
| **200 bank statements** | finds the duplicate charges hidden in about 3,500 charges | the duplicates were planted, so their ids are known | divide-and-conquer planning |

**Before you start:** you need an OpenRouter key in Colab's Secrets panel (the Setup section says how), and four files that you upload once (the section after Setup lists them).

---

### 🗺️ What You'll Do

1. **Measure the baseline with one call** per fee calculation: accuracy, cost and response time. (Exercise 1)
2. **Ask each question five times and keep the most common answer**, then see what the vote gained and what five calls cost. (Exercise 2)
3. **Split the reply to a complaint into a chain of three calls**, read its intermediate results, check each step with code, and add up what three calls in a row cost. (Exercise 3)
4. **Add a second call that reviews the first answer**, in two versions, and count the wrong answers it fixed and the right answers it broke. (Exercise 4)
5. **Have the model plan a long job**: find the duplicate charges in 200 statements with one call, then with a plan, one call per group and a limit of 30 calls. Then try the same planner on one simple question. (Exercise 5)
6. **Put every result in one table** and choose a technique for two situations. (Exercise 6)

After every exercise you read **the bill**: how many calls, what they cost, and what you have spent so far.

---

### 🧭 How to Use This Notebook

- **Task** notebook: fill the blanks (`___`), guided by the numbered `# TODO` comments. **Answer** notebook: the same, already filled in.
- Run the cells from top to bottom. Every exercise ends with ✅ / ❌ checks on the things your code controls. A 🤔 is not an error: it marks a result that depends on what the model answered in your run, with a note on how to read it.
- The model's answers differ a little from run to run, so your numbers will not match a classmate's exactly. That is expected, and Exercise 2 uses it.
- The whole notebook makes about 210 calls and takes about 4 minutes to run. Half of that time is one long call in Exercise 5. Calls are sent up to 8 at a time, so you do not wait for each one in turn.
- Total cost of this notebook: about **7 US cents**. The notebook prints the running total after every exercise.
- **Files to upload:** `fee_calculations.json`, `bank_fee_policy.md`, `customer_complaints.json` and `bank_statements.json`. You find them in the same place as this notebook.

💡 *Where the data comes from:* all four files were made for this course, and no real customer or bank is in them. The correct answer of every fee calculation is computed by code from the fee table. Every issue in a complaint is tied to one fact of the fee policy. The statements were generated by a script with a fixed starting number, which also planted the duplicate charges and stored their ids.

# 🔑 Setup: Your OpenRouter Key, Two Models and the Shared Helpers

**The key.** This notebook calls a hosted model through **OpenRouter**, one website that passes your request to the model you choose. It needs a secret named `OPENROUTER_API_KEY` in Colab's **🔑 Secrets** panel (left sidebar), with **Notebook access** switched on. If you do not have a key yet: create one at https://openrouter.ai/settings/keys, add a few dollars of credit, and save it as that secret. The steps are shown in detail in Unit 2 · Module 1 · Notebook 1: Your First Call.

**The two models.** A **token** is roughly a piece of a word. You pay for the tokens a model reads ("in") and for the tokens it writes ("out"). The prices below are in USD per million tokens, checked on 2026-11-14, and the cell looks them up again.

| Model | Price in | Price out | Used in | Why this one |
|---|---|---|---|---|
| `mistralai/ministral-8b-2512` | 0.15 | 0.15 | Exercises 1 to 4 | A small, fast model, the kind a team uses for high-volume work. It answers in about a second, and it makes mistakes on calculations with several steps, which gives the techniques something to fix. |
| `anthropic/claude-haiku-5.5` | 0.10 | 0.50 | Exercise 5 | A stronger model for the long job. Reading 3,500 charges carefully is beyond the small model: in our tests it reported hundreds of charges that are not duplicates. |

Two settings matter in this notebook:

- **Temperature** (the small model accepts it): how random the choice of each next token is. At `temperature=0` the model always takes its most likely token, so the same prompt gives almost the same answer every time. Every comparison in this notebook uses 0, so that a difference comes from the technique and not from chance. One exception is Exercise 2: a vote needs answers that differ, so it uses a normal temperature of 0.7. "Almost" is meant literally: with this model, a few answers still change between two runs at temperature 0.
- **Reasoning effort** (the stronger model has it): this model can think before it answers, which means it first writes notes to itself and you pay for them as output tokens. A setting says how much it may think. We set it to `low`, a normal setting for everyday work. It does not accept `temperature`, so no call to it sets one.

**The setup we recommend.** The results in this notebook were measured with these models and these settings, so run it this way first: it is where each technique's effect is easiest to see. How much a technique adds depends on the task and the model. On a harder task or a smaller model the difference is usually larger; on an easy task with a strong model it can be small.

**This cell also creates the helpers every exercise uses:**

- **`call_model(prompt)`:** sends one prompt to the small model (or, with `model=LONG_JOB_MODEL`, to the stronger one) and returns the reply with everything we measure: tokens, cost and seconds. If a call fails it tries once more, then writes the error down and moves on.
- **`run_in_parallel(function, items)`:** runs a function on many items, up to 8 at the same time, so 75 calls take about a minute and not ten.
- **`show_bill(calls, title)`:** prints the bill for a list of calls and the total you have spent so far.
- **`read_amount(text)`** and **`is_right(amount, expected)`:** read the number after `ANSWER:` in a reply, and check it against the correct answer to the cent.
- **`record_result(...)`:** writes one row into `RESULTS`, the table you complete in Exercise 6.

In [ ]:
# ==========================================================================
# 🔧 Setup: run this cell once, before any exercise
# ==========================================================================

# ==========================================================================
# Step 1: Install the libraries this notebook uses
# ==========================================================================
# Colab usually has them already; this line only adds what is missing.
# - openai: sends requests to OpenRouter (OpenRouter accepts the same request format as OpenAI's API)
%pip install -q openai

# ==========================================================================
# Step 2: Import the tools
# ==========================================================================
import json                                         # json: turns Python dictionaries into text and back
import math                                         # math: ceil() rounds a number up
import os                                           # os: checks whether a file exists
import re                                           # re: finds patterns in text, like "ANSWER: 12.50"
import textwrap                                     # textwrap: breaks long text into readable lines
import time                                         # time: measures how long each call takes
from collections import Counter                     # Counter: counts how often each value appears in a list
from concurrent.futures import ThreadPoolExecutor   # ThreadPoolExecutor: runs several calls at the same time
import pandas as pd                                 # pandas: tables of results
import requests                                     # requests: sends requests over the internet
from openai import OpenAI                           # OpenAI: the client we use to call models through OpenRouter
import os
# --- Local (VS Code) stand-ins for google.colab ---------------------------
from pathlib import Path
from dotenv import load_dotenv                      # load_dotenv: reads keys from the .env file next to this notebook
load_dotenv("D:/_Repos/_AIA/.env")

class userdata:  # replaces google.colab.userdata: reads the key from .env or an environment variable
    @staticmethod
    def get(name):
        value = os.environ.get(name)
        if not value:
            raise KeyError(f"{name} is not set in .env or the environment")
        return value

class files:  # replaces google.colab.files: VS Code has no upload picker, so files must sit in this folder
    @staticmethod
    def upload():
        print(f"   (local run) No upload picker here. Copy the file into: {Path.cwd()}")
        print("   Then run this cell again.")
        return {}
# ---------------------------------------------------------------------------

# ==========================================================================
# Step 3: Read your OpenRouter key from Colab Secrets
# ==========================================================================
# If the secret is missing, we stop with a message that says how to fix it.
try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    raise RuntimeError(
        "❌ No key found. Add OPENROUTER_API_KEY=<your key> to the .env file "
        "next to this notebook, then run this cell again."
    ) from None

# ==========================================================================
# Step 4: Choose the two models and look up their prices
# ==========================================================================
# To try another model later, change only its line here.
SMALL_MODEL = "mistralai/ministral-8b-2512"      # Exercises 1 to 4: small, fast, low cost
LONG_JOB_MODEL = "anthropic/claude-haiku-5.5"    # Exercise 5: a stronger model for the 200 statements

# OpenRouter publishes prices per single token; we multiply by 1,000,000 to get USD per million tokens.
# The values below are fallbacks (checked on 2026-11-14), used only if the lookup fails.
PRICES = {   # model id: (USD per million tokens in, USD per million tokens out)
    SMALL_MODEL: (0.15, 0.15),
    LONG_JOB_MODEL: (0.10, 0.50),
}
try:
    for model_info in requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]:
        if model_info["id"] in PRICES:
            PRICES[model_info["id"]] = (
                float(model_info["pricing"]["prompt"]) * 1_000_000,       # price of tokens the model reads
                float(model_info["pricing"]["completion"]) * 1_000_000,   # price of tokens the model writes
            )
        if model_info["id"] == LONG_JOB_MODEL and model_info.get("context_length"):
            LONG_JOB_CONTEXT_TOKENS = int(model_info["context_length"])
except Exception:
    print("⚠️ Could not look up the prices; using the fallback prices.")

# Reasoning effort of the stronger model: how much it may think before it answers ("low" = a little,
# a normal setting for everyday work). The small model has no such setting.
LONG_JOB_EFFORT = "low"
# How many tokens the stronger model can read in one call (its context window; checked on 2026-11-14)
LONG_JOB_CONTEXT_TOKENS = 1_000_000

# ==========================================================================
# Step 5: Create the client that sends our requests to OpenRouter
# ==========================================================================
# max_retries=0: the library does not retry by itself. call_model below retries once, so the measured time is honest.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY, max_retries=0)

# ==========================================================================
# Step 6: One function that makes a call and measures everything about it
# ==========================================================================
# Every call in this notebook goes through call_model. It records the reply, the tokens, the cost and the
# seconds, so each exercise can add them up.
ALL_CALLS = []              # every call this notebook makes is written down here (used for the running total)
CALL_TIMEOUT_SECONDS = 300  # give up on a call after 5 minutes (the longest call of this notebook takes about 2)


def call_model(prompt, max_tokens=1500, temperature=0, model=None):
    """Send one prompt to a model. Return a dictionary with the reply and what we measure. Never raises an error.

    max_tokens is the most tokens the model may write. It is set high enough that an answer is never cut off.
    temperature is used for the small model only: 0 = always take the most likely next token.
    model: leave it out for the small model; pass LONG_JOB_MODEL for the stronger one.
    """
    model = model or SMALL_MODEL
    request = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "timeout": CALL_TIMEOUT_SECONDS,
    }
    if model == SMALL_MODEL:
        request["temperature"] = temperature
    else:
        # The stronger model does not accept temperature. It gets its reasoning effort instead.
        # extra_body carries settings that only OpenRouter knows. "reasoning" is its word for thinking.
        request["extra_body"] = {"reasoning": {"effort": LONG_JOB_EFFORT}}
    record = {"text": "", "tokens_in": 0, "tokens_out": 0, "cost": 0.0, "seconds": 0.0,
              "finish_reason": None, "error": None}
    for attempt in (1, 2):   # one try, and one retry if the first try fails
        start = time.perf_counter()   # perf_counter: a precise clock for measuring how long something takes
        try:
            # **request hands every key of the dictionary to the function as a named setting
            response = client.chat.completions.create(**request)
            record["seconds"] = time.perf_counter() - start   # the response time of this call
            if not getattr(response, "choices", None):        # OpenRouter answered, but without a reply from the model
                raise RuntimeError(f"no reply in the response: {getattr(response, 'error', None)}")
            choice = response.choices[0]
            record["text"] = (choice.message.content or "").strip()
            record["finish_reason"] = choice.finish_reason    # "stop" = finished; "length" = cut off at max_tokens
            usage = response.usage.model_dump()               # the token counts of this call, as a dictionary
            record["tokens_in"] = usage.get("prompt_tokens") or 0        # tokens the model read
            record["tokens_out"] = usage.get("completion_tokens") or 0   # tokens the model wrote (thinking included, if it thinks)
            # OpenRouter reports what it charged for the call in usage["cost"] (in credits; one credit is one US dollar).
            # If that field is missing, we work the cost out from the token counts and the price per million tokens.
            billed = usage.get("cost")
            if billed is None:
                price_in, price_out = PRICES[model]
                billed = (record["tokens_in"] * price_in + record["tokens_out"] * price_out) / 1_000_000
            record["cost"] = float(billed)
            record["error"] = None
            break   # the call worked: no retry needed
        except Exception as error:
            record["seconds"] = time.perf_counter() - start
            record["error"] = f"{type(error).__name__}: {str(error)[:200]}"   # [:200] keeps the message short
            if attempt == 1:
                time.sleep(2)   # wait two seconds, then try once more
    ALL_CALLS.append(record)
    return record


# ==========================================================================
# Step 7: Run many calls at the same time
# ==========================================================================
# A call spends most of its time waiting for the model. While one call waits, others can be sent.
# This only saves YOUR waiting time in the notebook: every call is still timed by itself and billed by itself.
def run_in_parallel(function, items, workers=8):
    """Run function(item) for every item, up to `workers` at the same time. Return the results in the same order."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(function, items))


# ==========================================================================
# Step 8: The bill for a list of calls
# ==========================================================================
def show_bill(calls, title):
    """Print the number of calls, the tokens and the cost of a list of calls, and the total spent so far."""
    cost = sum(call["cost"] for call in calls)
    tokens_in = sum(call["tokens_in"] for call in calls)
    tokens_out = sum(call["tokens_out"] for call in calls)
    print(f"🧾 Bill for {title}: {len(calls)} calls · {tokens_in:,} tokens in · {tokens_out:,} tokens out · ${cost:.4f}")
    failed = [call for call in calls if call["error"]]
    if failed:
        print(f"   ⚠️ {len(failed)} of these calls failed twice and have no reply. First error: {failed[0]['error']}")
    print(f"   Spent so far in this notebook: ${sum(call['cost'] for call in ALL_CALLS):.4f} ({len(ALL_CALLS)} calls)")
    return cost


# ==========================================================================
# Step 9: Read the number in a reply and check it
# ==========================================================================
# The calculation prompts ask the model to end with a line like "ANSWER: 12.50". These two helpers
# find that number and compare it with the correct answer.
def read_amount(text):
    """Return the number after the last 'ANSWER:' in the text (12.5 for 'ANSWER: $12.50'), or None if there is none."""
    # The pattern: "ANSWER:", then optional spaces, markdown stars and a dollar sign, then the number itself
    found = re.findall(r"ANSWER:\s*[*_`]*\s*\$?\s*(-?[\d,]*\d(?:\.\d+)?)", text, re.IGNORECASE)
    if not found:
        return None
    return float(found[-1].replace(",", ""))   # [-1] is the last match; "1,250.00" becomes 1250.0


def is_right(amount, expected):
    """True when the amount is the expected one, to the cent."""
    return amount is not None and abs(amount - expected) < 0.005


def as_dollars(amount):
    """A number as text: 12.5 becomes '$12.50'; None becomes 'no number'."""
    return "no number" if amount is None else f"${amount:,.2f}"


def times_as_much(value, base):
    """How many times `base` fits into `value`, as text ('5.0x'). Avoids dividing by zero."""
    return f"{value / base:.1f}x" if base else "n/a"


# ==========================================================================
# Step 10: The results table that grows through the notebook
# ==========================================================================
# Each exercise adds its rows here. Exercise 6 turns the list into one table.
RESULTS = []


def record_result(task, technique, result, accuracy, calls, cost, seconds, seconds_parallel=None):
    """Store one row: what a technique achieved on a task and what ONE answer cost.

    calls, cost and seconds are for one answer (one calculation, one reply, or one whole job on the statements).
    seconds: the waiting time when the calls run one after another.
    seconds_parallel: the waiting time when calls that do not depend on each other run at the same time.
    """
    row = {"task": task, "technique": technique, "result": result, "accuracy": accuracy, "calls": calls,
           "cost": cost, "seconds": seconds,
           "seconds_parallel": seconds if seconds_parallel is None else seconds_parallel}
    # Running a cell twice replaces its row; it does not add a second one
    RESULTS[:] = [old for old in RESULTS if (old["task"], old["technique"]) != (task, technique)]
    RESULTS.append(row)
    print(f"📌 Stored for the final table: {task} · {technique} · {result} · {calls} call(s) · "
          f"${cost:.5f} · {seconds:.1f} s per answer")


# ==========================================================================
# Step 11: Helpers for the files you upload
# ==========================================================================
# A file counts as "here" only if it exists and really opens: a broken or half-uploaded file is asked for again.
def file_is_ready(name):
    """True if the file exists and opens as its type (JSON here); other types: not empty."""
    if not os.path.exists(name) or os.path.getsize(name) == 0:
        return False
    try:
        if name.lower().endswith(".json"):
            with open(name) as f:
                json.load(f)
        return True
    except Exception:
        return False


def describe_file(name):
    """One short line on what a file holds, so you can see the upload worked."""
    size_kb = os.path.getsize(name) / 1024
    if name.lower().endswith(".json"):
        with open(name) as f:
            data = json.load(f)
        what = f"{len(data)} entries" if isinstance(data, list) else f"{len(data)} top-level keys"
        return f"{what}, {size_kb:.1f} KB"
    return f"{size_kb:.1f} KB"


print("✅ Setup done.")
for model_id, used_for in ((SMALL_MODEL, "Exercises 1 to 4"), (LONG_JOB_MODEL, "Exercise 5")):
    print(f"   {model_id} ({used_for}): ${PRICES[model_id][0]:.2f} per million tokens in, "
          f"${PRICES[model_id][1]:.2f} per million tokens out")

# 📁 Upload the Files (Optional, Recommended)

This notebook uses **4 files** that we give you. You find them in the same place as this notebook. Upload them once now, and every exercise will find them.

### 📄 The files

1. **`fee_calculations.json`**: used in Exercises 1, 2, 4 and 5
   - **Contains:** the bank's fee table and 15 fee calculations that need several steps
   - **Fields:** `fee_table`; for each calculation `id`, `question`, `answer` (in dollars), `steps` (the worked solution)
2. **`bank_fee_policy.md`**: used in Exercise 3
   - **Contains:** the bank's fee policy for personal accounts, in 12 numbered sections (about three pages)
3. **`customer_complaints.json`**: used in Exercise 3
   - **Contains:** 20 customer complaints, each with three different issues
   - **Fields:** `id`, `tone`, `message`; for each issue `issue`, `section`, `fact` (the policy fact that answers it), `check` (patterns a good reply must contain)
4. **`bank_statements.json`**: used in Exercise 5
   - **Contains:** 200 bank statements, one customer's month each, with duplicate charges planted in some of them
   - **Fields:** `duplicate_rule`; for each statement `statement_id`, `period_start`, `period_end`, `transactions` (`id`, `date`, `merchant`, `amount`); `planted_duplicates` (the ids of the planted duplicates)

### 🎯 Two ways to do it

- **Option 1 (recommended):** run the cell below and upload all the files when Colab asks. The exercises then run without interruptions, and you see right away that every file opened.
- **Option 2:** skip this cell. Each exercise asks for its own file when it needs it.

**Note:** uploaded files live in this Colab session. Restarting the runtime keeps them; when the runtime disconnects, they are deleted and you upload them again (this cell tells you which ones).

In [ ]:
# ==========================================================================
# 📁 Upload the Files (optional: each exercise also asks for its own file)
# ==========================================================================

# Make sure the Setup cell ran first (it creates file_is_ready and describe_file)
assert "file_is_ready" in globals(), "Run the Setup cell at the top of the notebook first."

# Every file this notebook needs, and the exercise that uses it
REQUIRED_FILES = {
    "fee_calculations.json": "Exercises 1, 2, 4 and 5: the fee table and the 15 calculations",
    "bank_fee_policy.md": "Exercise 3: the fee policy",
    "customer_complaints.json": "Exercise 3: the 20 complaints",
    "bank_statements.json": "Exercise 5: the 200 statements",
}

# ==========================================================================
# Step 1: Check which files are already here
# ==========================================================================
# Running this cell again is safe: files that are here and open are reused, not asked for again.
ready = [name for name in REQUIRED_FILES if file_is_ready(name)]
missing = [name for name in REQUIRED_FILES if name not in ready]

print("=" * 70)
print("📁 FILES FOR THIS NOTEBOOK")
print("=" * 70)
for name in ready:
    print(f"✅ Already here: {name} ({describe_file(name)})")
for name in missing:
    print(f"📂 Missing: {name} (needed for {REQUIRED_FILES[name]})")

# ==========================================================================
# Step 2: Upload the missing files
# ==========================================================================
if missing:
    print("\n👉 Click 'Choose Files' and select the missing files")
    uploaded = files.upload()   # opens Colab's file picker; returns {file name: contents}
    for name in uploaded:
        if name in REQUIRED_FILES:
            print(f"✅ Uploaded: {name}")
        else:
            # A browser often renames a second download, e.g. "bank_statements (1).json"
            print(f"⚠️ Uploaded '{name}', but this notebook expects: {', '.join(missing)}. "
                  "Rename the file on your computer and upload it again.")

    # ======================================================================
    # Step 3: Check again, by opening every file
    # ======================================================================
    still_missing = [name for name in missing if not file_is_ready(name)]
    print()
    for name in missing:
        if name not in still_missing:
            print(f"📋 {name}: {describe_file(name)}")
    if still_missing:
        print("⚠️ Still missing or not opening: " + ", ".join(still_missing))
        print("   Each exercise will ask for its file again when it needs it.")
    else:
        print("🎉 All files uploaded and checked. Every exercise can run without asking again.")
else:
    print("\n🎉 All files are already here. No upload needed.")

# 📏 Topic 1: One Call as the Baseline: Accuracy, Cost and Response Time

Every technique in this notebook is "the same job, done with more calls". Before you can say whether more calls are worth it, you need to know what **one** call achieves. That first measurement is called the **baseline**: the plain version that every later version is compared with.

Here is one of the jobs. A customer on the Standard plan sent three instant transfers today: \$30, \$400 and \$2,500. What do they pay in transfer fees? The fee table says an instant transfer costs 1% of the amount, with a minimum of \$0.50 and a maximum of \$10. So the answer needs three small steps, and two of them have a trap:

| Transfer | 1% of the amount | The rule that applies | Fee |
|---|---|---|---|
| \$30 | \$0.30 | below the minimum | \$0.50 |
| \$400 | \$4.00 | neither | \$4.00 |
| \$2,500 | \$25.00 | above the maximum | \$10.00 |
| | | **Total** | **\$14.50** |

A model that answers quickly can forget the minimum and say \$14.30, or forget both limits and say \$29.30. That is why calculations with several steps make a good test: there is exactly one right number, and code can check it.

For a single call you measure three numbers, and you will measure the same three for every technique:

- **Accuracy:** how many of the answers are right. Here: how many of 15 calculations come back with the correct total, to the cent.
- **Cost:** what the calls cost in dollars. You pay for the tokens the model reads and for the tokens it writes.
- **Response time:** how many seconds pass between sending the request and having the answer. Engineers also call it **latency**. For a customer in a chat, this is the time they spend looking at a spinner.

One number alone tells you little. A technique that raises accuracy from 80% to 90% sounds good until you see that it costs five times as much and makes the customer wait five times as long. The three numbers together let you decide.

---

### 🔍 What You'll Learn

- Send the same kind of prompt for 15 questions and check every answer with code
- Measure accuracy, cost and response time for a set of calls
- Store a baseline so that later versions can be compared with it

### 🧠 Why This Matters

- Without a baseline you cannot tell whether a technique helped: "14 of 15" means nothing until you know the plain call got 11
- Cost and response time decide whether a technique can be used at all: a correct answer that arrives after the customer left is of no use in a chat

## 🧪 Exercise 1: Measure the Baseline With One Call

You give the model each of the 15 fee calculations in a **single call**: the fee table, the question, and a request to work the answer out briefly and to finish with one line, `ANSWER: <amount>`. A provided function reads the number and compares it with the correct answer. You then work out three numbers: the accuracy, the cost of the 15 calls, and the average response time.

These three numbers are the baseline of this notebook. Exercises 2 and 4 run techniques on the same 15 calculations and put their numbers next to these.

The calls go to the small model at temperature 0. A small model is a realistic choice for a job like this, and it is not perfect at it: in our runs it got between 6 and 10 of the 15 calculations right. That gap is what the techniques in Exercises 2 and 4 try to close. A baseline that is already perfect would leave nothing to measure.

### 🎯 What You'll Do

1. Load the fee table and the 15 calculations, and read one of them with its worked solution.
2. Write the function that answers one calculation with one call.
3. Run it on all 15 calculations and look at each answer next to the correct one.
4. Work out the accuracy, the total cost and the average response time, and store them as the baseline.

👉 By the end, you'll have three baseline numbers that every later technique is compared with.

---

### ✅ You Succeed When

- You have 15 answers, one per calculation
- `BASELINE` holds the accuracy, the cost and the average response time, and each matches the 15 answers
- The baseline row is stored in `RESULTS`

---

### 🛠 Technical Info

- **One call:** `call_model(prompt, max_tokens=...)` returns a dictionary: `call["text"]` (the reply), `call["cost"]` (dollars) and `call["seconds"]` (the response time)
- **Reading the answer:** `read_amount(text)` returns the number after `ANSWER:`, or `None`; `is_right(amount, expected)` compares to the cent
- **Many calls:** `run_in_parallel(function, items)` returns one result per item, in the same order
- **Honest note:** the 15 calls are sent up to 8 at a time, so you wait a few seconds and not half a minute. Each call is still timed by itself with `time.perf_counter()`, so the average is the time one customer would wait for one answer. Even at temperature 0 this model's answers are not perfectly repeatable: run the cell twice and the baseline can move by one or two answers.
- **Difficulty:** 🟢 Level 1, Guided

---

### 🚀 Level Up (optional)

1. Send the same 15 prompts to the stronger model (`call_model(..., model=LONG_JOB_MODEL)`) and compare the three numbers. A stronger model is one more way to buy accuracy with cost.
2. Print the model's full reply for one wrong answer and work out which rule of the fee table it missed, using the `steps` of that calculation.

In [ ]:
# ==========================================================================
# 📏 Exercise 1: Measure the Baseline With One Call
# ==========================================================================

# Make sure the Setup cell ran first (it creates call_model and the other helpers)
assert "call_model" in globals(), "Run the Setup cell at the top of the notebook first."

# ==========================================================================
# Step 1: Load the fee table and the 15 calculations (upload the file if it is not here yet)
# ==========================================================================
FEE_FILE = "fee_calculations.json"
if not file_is_ready(FEE_FILE):
    print(f"📁 Please upload '{FEE_FILE}':")
    files.upload()
assert file_is_ready(FEE_FILE), f"'{FEE_FILE}' is missing or does not open. Upload it again (the exact name matters)."
print(f"📋 {FEE_FILE}: {describe_file(FEE_FILE)}")

with open(FEE_FILE) as f:
    fee_data = json.load(f)
FEE_TABLE = fee_data["fee_table"]          # the bank's fees as one table (text)
CALCULATIONS = fee_data["calculations"]    # a list of 15 dictionaries: id, question, answer, steps

# ==========================================================================
# Step 2: Look at the data before using it
# ==========================================================================
# The first lines of the fee table, and one calculation with its worked solution.
print()
print("=" * 70)
print("📊 THE FEE TABLE (first 6 lines) AND ONE CALCULATION")
print("=" * 70)
print("\n".join(FEE_TABLE.split("\n")[:6]))
print("...")
example = CALCULATIONS[2]
print(f"\n{example['id']}: " + textwrap.fill(example["question"], width=100))
for step in example["steps"]:
    print(f"   step: {step}")
print(f"   correct answer: {as_dollars(example['answer'])}")

# ==========================================================================
# Step 3: Answer one calculation with one call
# ==========================================================================
# The prompt has three parts: the fee table, the question, and the form of the answer.
# The tags (<fee_table>, <question>) only mark where each part starts and ends.
ANSWER_MAX_TOKENS = 1500   # room for the working plus the answer line, so nothing is cut off


def calculation_prompt(question):
    """The prompt for one fee calculation: fee table, question, and the form of the answer."""
    return (
        "You work in a bank's support team. Use the fee table to answer the question.\n\n"
        f"<fee_table>\n{FEE_TABLE}</fee_table>\n\n"
        f"<question>\n{question}\n</question>\n\n"
        "Work it out briefly, then finish with one last line in exactly this form "
        "(the total in US dollars, with two decimals):\n"
        "ANSWER: 12.50"
    )


def answer_once(calculation, temperature=0):
    """One call for one calculation. Return the amount, whether it is right, and the call (reply, cost, seconds)."""
    # temperature=0: always take the most likely next token, so the answer is (almost) the same every time
    # TODO 1: Send the prompt for this calculation's question. Hint: calculation_prompt(calculation["question"])
    call = call_model(___, max_tokens=ANSWER_MAX_TOKENS, temperature=temperature)  # <-- Fill in the blank
    # TODO 2: Read the number after "ANSWER:" in the reply. Hint: the reply is call["text"]
    amount = read_amount(___)  # <-- Fill in the blank
    return {
        "id": calculation["id"],
        "amount": amount,
        "right": is_right(amount, calculation["answer"]),
        "call": call,
    }


# ==========================================================================
# Step 4: Run the single call on all 15 calculations
# ==========================================================================
# Up to 8 calls are sent at the same time. The results come back in the order of CALCULATIONS.
baseline_answers = run_in_parallel(answer_once, CALCULATIONS)

print()
print("=" * 70)
print("📊 ONE CALL PER CALCULATION")
print("=" * 70)
print(f"{'id':<5}{'correct':>10}{'model':>12}   {'':<6}{'seconds':>8}")
for calculation, answer in zip(CALCULATIONS, baseline_answers):   # zip walks through both lists side by side
    mark = "right" if answer["right"] else "wrong"
    print(f"{answer['id']:<5}{as_dollars(calculation['answer']):>10}{as_dollars(answer['amount']):>12}   "
          f"{mark:<6}{answer['call']['seconds']:>8.1f}")

# ==========================================================================
# Step 5: The three baseline numbers
# ==========================================================================
# Accuracy: the share of right answers. Cost: all 15 calls together. Response time: the average of the 15 calls.
baseline_calls = [answer["call"] for answer in baseline_answers]
right_count = sum(answer["right"] for answer in baseline_answers)   # True counts as 1, False as 0
BASELINE = {
    "right": right_count,
    # TODO 3: The share of right answers: the number of right answers divided by the number of answers
    "accuracy": ___,  # <-- Fill in the blank
    "cost": sum(call["cost"] for call in baseline_calls),
    # TODO 4: The average response time: add up call["seconds"] of every call, then divide by the number of calls
    "seconds": ___,  # <-- Fill in the blank
}

print()
print("=" * 70)
print("📊 THE BASELINE")
print("=" * 70)
print(f"Accuracy              : {BASELINE['right']} of {len(CALCULATIONS)} right ({BASELINE['accuracy']:.0%})")
print(f"Cost of the 15 calls  : ${BASELINE['cost']:.4f}")
print(f"Average response time : {BASELINE['seconds']:.1f} seconds per answer")
show_bill(baseline_calls, "Exercise 1")

# ==========================================================================
# Step 6: Store the baseline for the final table
# ==========================================================================
# One answer takes 1 call. Its cost is the total divided by 15.
record_result(
    task="15 fee calculations", technique="single call",
    result=f"{BASELINE['right']} of {len(CALCULATIONS)} right", accuracy=BASELINE["accuracy"],
    calls=1, cost=BASELINE["cost"] / len(CALCULATIONS), seconds=BASELINE["seconds"],
)

# ==========================================================================
# Step 7: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ You have 15 answers, one per calculation" if len(baseline_answers) == 15 and
      [answer["id"] for answer in baseline_answers] == [calculation["id"] for calculation in CALCULATIONS]
      else f"❌ Expected 15 answers in the order of CALCULATIONS, got {len(baseline_answers)}")
expected_accuracy = sum(is_right(answer["amount"], calculation["answer"])
                        for calculation, answer in zip(CALCULATIONS, baseline_answers)) / 15
expected_cost = sum(answer["call"]["cost"] for answer in baseline_answers)
expected_seconds = sum(answer["call"]["seconds"] for answer in baseline_answers) / 15
numbers_match = (abs(BASELINE["accuracy"] - expected_accuracy) < 1e-9 and abs(BASELINE["cost"] - expected_cost) < 1e-9
                 and abs(BASELINE["seconds"] - expected_seconds) < 1e-9)
print("✅ BASELINE holds the accuracy, the cost and the average response time of the 15 answers" if numbers_match
      else "❌ BASELINE does not match the 15 answers: check the three formulas in Step 5")
print("✅ The baseline row is stored in RESULTS" if any(
      row["task"] == "15 fee calculations" and row["technique"] == "single call" for row in RESULTS)
      else "❌ The baseline row is missing in RESULTS")
no_number = sum(answer["amount"] is None for answer in baseline_answers)
if no_number:
    print(f"🤔 {no_number} replies had no 'ANSWER:' line with a number. They count as wrong. If a call failed "
          "(see the bill above), run the cell again.")
if BASELINE["right"] == len(CALCULATIONS):
    print("🤔 The single call got all 15 right in your run, which is unusual for this small model. The techniques "
          "then have no accuracy to gain here, and you will see only what they cost.")
else:
    print(f"🤔 The single call got {len(CALCULATIONS) - BASELINE['right']} of 15 wrong in your run. "
          "This number moves by one or two from run to run. Those wrong answers are what the techniques try to fix.")
print("🧠 A baseline is three numbers, not one: how often it is right, what it costs, and how long it takes.")

# 🗳️ Topic 2: Self-Consistency, or Asking Several Times and Keeping the Most Common Answer

Ask a model the same question twice and you can get two different answers. Each next token is picked with some chance, so an answer to a hard question sometimes takes a wrong turn. How much chance is involved is set by **temperature**: at 0 the model always takes its most likely token, at a normal value such as 0.7 less likely tokens get picked too. **Self-consistency** turns that weakness into a tool: you ask the same question several times, and you keep the answer that comes back most often. Choosing the most common answer is called a **majority vote**.

Take the three instant transfers from Topic 1 (correct total: \$14.50) and ask five times:

| Call | Answer | What happened |
|---|---|---|
| 1 | \$14.50 | right |
| 2 | \$29.30 | forgot the minimum and the maximum |
| 3 | \$14.50 | right |
| 4 | \$14.30 | forgot the minimum |
| 5 | \$14.50 | right |

The vote is \$14.50, with 3 of 5 answers. Why does this work? There is one right answer and many different wrong ones. When the model slips, it slips in different places, so the wrong answers scatter: \$29.30 here, \$14.30 there. The right answers all land on the same number, so they pile up.

In code, the whole technique is a loop and a count:

```python
answers = [ask(question) for _ in range(5)]            # the same question, five times
winner, votes = Counter(answers).most_common(1)[0]     # the most common answer and how often it came
```

It used to be described as a prompt technique. Today it is these two lines around the model.

The five answers must be able to differ, so the calls of a vote use a normal temperature, not 0. Five calls at temperature 0 would give (almost) the same answer five times, and a vote over identical answers changes nothing.

Two more things come with it:

- **The vote tells you how sure to be.** Five identical answers and a 3-to-2 split are both "an answer", but they are not equally trustworthy. The number of answers that agree is a free warning signal: a question where the answers disagree is a question the model finds hard.
- **It cannot fix a mistake the model makes every time.** If the model misreads a rule, it misreads it in all five calls, and the vote confirms the wrong answer with 5 of 5. Voting removes random slips, not a wrong understanding.

The price is simple: five calls are five times the work. The bill is up to five times as high: some providers charge less for a prompt they have read a moment ago (this is called **prompt caching**), and the five calls of a vote send the same prompt. If they run one after another, the customer also waits five times as long. They do not depend on each other, so they can run at the same time, and then the wait is the time of the slowest of the five.

---

### 🔍 What You'll Learn

- Build self-consistency: several calls for one question and a majority vote
- Use the agreement between the answers as a sign of how hard a question is
- Work out what a vote costs in calls, money and waiting time, one after another and in parallel

### 🧠 Why This Matters

- For a number or a label that must be right (a fee, a diagnosis code, a yes or no), a vote is one of the few ways to buy accuracy without changing the model or the prompt
- If you do not look at the agreement, you throw away a signal you already paid for: the questions to send to a person are the ones where the answers split

## 🧪 Exercise 2: Ask Each Question Five Times and Keep the Most Common Answer

You take the same 15 fee calculations and the same prompt as in Exercise 1. This time you call the model **five times per calculation**, at temperature 0.7 so that the answers can differ, and keep the answer that appears most often. That is 75 calls.

Then you put the result next to the baseline from Exercise 1 (one call at temperature 0), in all three numbers. The accuracy: did the vote fix some of the answers the single call got wrong? You also see how often a single one of the 75 sampled answers is right, so you can tell what the voting itself adds. The number of calls: five times the baseline, by construction. The cost: up to five times the baseline. The response time: five times as long if the five calls run one after another, or as long as the slowest call if they run at the same time. The notebook reports both.

Last, you look at where the five answers disagreed. If the idea of Topic 2 holds, the questions with a split vote should be the ones the single call got wrong in Exercise 1.

### 🎯 What You'll Do

1. Call the model five times for each of the 15 calculations.
2. Write the vote: the most common of five answers, and how many answers agreed.
3. Compare the accuracy of the vote with the baseline.
4. Work out what the vote cost: calls, dollars, and seconds per answer (one after another, and in parallel).
5. List the questions by how much the five answers disagreed, next to what the single call did on each.

👉 By the end, you'll know what five calls bought on these 15 questions and what they cost.

---

### ✅ You Succeed When

- You have 75 answers: 5 for each of the 15 calculations
- Every vote is the most common of its five answers
- The vote used exactly 5 times the calls of the baseline
- The self-consistency row is stored in `RESULTS`

---

### 🛠 Technical Info

- **Counting:** `Counter(list).most_common(1)` returns a list with one pair: `[(value, how_often)]`
- **Reuse:** `answer_once(calculation, temperature=...)` from Exercise 1 makes one call and returns `amount`, `right` and `call`
- **Honest note:** the baseline is one call at temperature 0, and the five answers of a vote are sampled at 0.7. That is how the two would be used in practice, and it means the comparison has two parts: a single sampled answer is not better than the baseline call, and the vote over five of them is. In our eleven runs the single call got 6 to 10 of 15 right and the vote 9 to 13. The vote was ahead by 1 to 4 answers in ten runs and level in one. With 15 questions your numbers will differ from ours. The first answer of Exercise 1 is not reused here: all five answers are new calls.
- **Honest note about the cost:** the vote makes exactly five times the calls, but the bill is not exactly five times the baseline (in our runs between 2.6 and 5.3 times). The provider of this model charges less for the part of a prompt it has just read (prompt caching), and which calls get that discount changes from run to run. The notebook shows what OpenRouter really charged.
- **Honest note about the time:** the notebook sends the 75 calls up to 8 at a time. "One after another" is the sum of the five measured call times, and "in parallel" is the longest of the five. Both are worked out from the same measurements.
- **Difficulty:** 🟢 Level 1, Guided

---

### 🚀 Level Up (optional)

1. Set `VOTES = 3` and run the cell again. Does a vote of three fix as many answers as a vote of five, for 60% of the cost?
2. Call only once more when the first two answers agree, and up to five times when they do not. Count how many calls that saves on these 15 questions.

In [ ]:
# ==========================================================================
# 🗳️ Exercise 2: Ask Each Question Five Times and Keep the Most Common Answer
# ==========================================================================

# Make sure the Setup cell and Exercise 1 ran first (Exercise 1 creates answer_once and BASELINE)
assert "call_model" in globals(), "Run the Setup cell at the top of the notebook first."
assert "BASELINE" in globals(), "Run Exercise 1 first: this exercise compares with its baseline."

# ==========================================================================
# Step 1: Ask every question five times
# ==========================================================================
# We build one long list: each calculation five times in a row (75 items), and send it through answer_once.
# A vote needs answers that differ, so these calls use a normal temperature: at 0.7 the model sometimes
# picks a less likely next token, and two calls can take different turns.
# TODO 1: Ask each question five times, at a temperature of 0.7 so that the answers can differ
VOTES = ___  # <-- Fill in the blank (a number)
VOTE_TEMPERATURE = ___  # <-- Fill in the blank (a number)
repeated_questions = [calculation for calculation in CALCULATIONS for _ in range(VOTES)]
# lambda calculation: ... is a small function without a name; it calls answer_once at the vote's temperature
all_answers = run_in_parallel(lambda calculation: answer_once(calculation, temperature=VOTE_TEMPERATURE),
                              repeated_questions)

# Cut the 75 answers back into 15 groups of five: answers 0 to 4 belong to the first question, 5 to 9 to the second
answers_per_question = [all_answers[start:start + VOTES] for start in range(0, len(all_answers), VOTES)]


# ==========================================================================
# Step 2: The vote
# ==========================================================================
# Counter counts how often each amount appears. most_common(1) gives the amount with the highest count.
# A reply without a number (None) is left out of the count. If two amounts tie, the one that came first wins.
def majority_vote(amounts):
    """Return (the most common amount, how many answers gave it). (None, 0) if no answer had a number."""
    counted = Counter(amount for amount in amounts if amount is not None)
    if not counted:
        return None, 0
    # TODO 2: Take the most common amount and its count. Hint: counted.most_common(1) returns [(amount, count)]
    winner, how_often = counted.___(1)[0]  # <-- Fill in the blank
    return winner, how_often


votes = []
for calculation, answers in zip(CALCULATIONS, answers_per_question):
    amounts = [answer["amount"] for answer in answers]
    winner, agreement = majority_vote(amounts)
    votes.append({
        "id": calculation["id"],
        "amounts": amounts,
        "winner": winner,
        "agreement": agreement,                                   # 5 = all five agree, 3 = a 3-to-2 split
        "right": is_right(winner, calculation["answer"]),
        # TODO 3: The waiting time of five calls. One after another, the times add up. At the same time,
        # you wait for the slowest call. Use sum(...) for one and max(...) for the other.
        "seconds_in_a_row": ___(answer["call"]["seconds"] for answer in answers),  # <-- Fill in the blank
        "seconds_in_parallel": ___(answer["call"]["seconds"] for answer in answers),  # <-- Fill in the blank
    })

print("=" * 70)
print("📊 FIVE ANSWERS AND THE VOTE")
print("=" * 70)
print(f"{'id':<5}{'correct':>9}   {'the five answers':<48}{'vote':>9}  {'agree':<7}{'vote is'}")
for calculation, vote in zip(CALCULATIONS, votes):
    five = "  ".join(as_dollars(amount) for amount in vote["amounts"])
    print(f"{vote['id']:<5}{as_dollars(calculation['answer']):>9}   {five:<48}{as_dollars(vote['winner']):>9}  "
          f"{vote['agreement']} of {VOTES}  {'right' if vote['right'] else 'wrong'}")

# ==========================================================================
# Step 3: Compare the accuracy with the baseline
# ==========================================================================
vote_right = sum(vote["right"] for vote in votes)
vote_accuracy = vote_right / len(votes)
sampled_right = sum(answer["right"] for answer in all_answers)   # how many of the 75 single answers are right
# TODO 4: "fixed" lists the questions the vote got right and the single call got wrong.
# Hint: vote["right"] and single["right"] are True or False; use "not" for the one that must be wrong
fixed = [vote["id"] for vote, single in zip(votes, baseline_answers) if ___ and ___]  # <-- Fill in the blanks
broken = [vote["id"] for vote, single in zip(votes, baseline_answers) if not vote["right"] and single["right"]]

# ==========================================================================
# Step 4: What the vote cost
# ==========================================================================
# Calls and cost are for all 15 questions. The two times are averages per answer.
vote_calls = [answer["call"] for answer in all_answers]
vote_cost = sum(call["cost"] for call in vote_calls)
vote_seconds_in_a_row = sum(vote["seconds_in_a_row"] for vote in votes) / len(votes)
vote_seconds_in_parallel = sum(vote["seconds_in_parallel"] for vote in votes) / len(votes)

print()
print("=" * 70)
print("📊 SINGLE CALL AGAINST THE VOTE OF FIVE")
print("=" * 70)
print(f"{'':<34}{'single call':>14}{'vote of 5':>14}{'vote / single':>16}")
print(f"{'Right answers (of 15)':<34}{BASELINE['right']:>14}{vote_right:>14}")
print(f"{'Share of right answers':<34}{BASELINE['accuracy']:>14.0%}{vote_accuracy:>14.0%}")
print(f"{'Calls for the 15 questions':<34}{len(baseline_calls):>14}{len(vote_calls):>14}"
      f"{times_as_much(len(vote_calls), len(baseline_calls)):>16}")
print(f"{'Cost of the 15 questions':<34}{'$' + format(BASELINE['cost'], '.4f'):>14}{'$' + format(vote_cost, '.4f'):>14}"
      f"{times_as_much(vote_cost, BASELINE['cost']):>16}")
print(f"{'Seconds per answer, one by one':<34}{BASELINE['seconds']:>14.1f}{vote_seconds_in_a_row:>14.1f}"
      f"{times_as_much(vote_seconds_in_a_row, BASELINE['seconds']):>16}")
print(f"{'Seconds per answer, in parallel':<34}{BASELINE['seconds']:>14.1f}{vote_seconds_in_parallel:>14.1f}"
      f"{times_as_much(vote_seconds_in_parallel, BASELINE['seconds']):>16}")
print(f"\nOne sampled answer at temperature {VOTE_TEMPERATURE}, without a vote: {sampled_right} of {len(all_answers)} "
      f"right ({sampled_right / len(all_answers):.0%})")
print(f"Wrong with one call, right with the vote: {', '.join(fixed) or 'none'}")
print(f"Right with one call, wrong with the vote: {', '.join(broken) or 'none'}")
show_bill(vote_calls, "Exercise 2")

# ==========================================================================
# Step 5: Where did the five answers disagree?
# ==========================================================================
# We sort the questions by agreement, lowest first, and put the single call's result from Exercise 1 next to each.
# "Split" means: not all five answers were the same.
print()
print("=" * 70)
print("📊 DISAGREEMENT AGAINST THE SINGLE CALL'S ERRORS")
print("=" * 70)
by_agreement = sorted(zip(votes, baseline_answers), key=lambda pair: pair[0]["agreement"])
for vote, single in by_agreement:
    different = len(set(vote["amounts"]))   # set() keeps each different answer once
    print(f"{vote['id']}: {vote['agreement']} of {VOTES} agree, {different} different answer(s) · "
          f"single call in Exercise 1: {'right' if single['right'] else 'wrong'}")

split = [(vote, single) for vote, single in by_agreement if vote["agreement"] < VOTES]
unanimous = [(vote, single) for vote, single in by_agreement if vote["agreement"] == VOTES]
split_wrong = sum(not single["right"] for _, single in split)
unanimous_wrong = sum(not single["right"] for _, single in unanimous)
print(f"\nQuestions with a split vote : {len(split)}, and the single call was wrong on {split_wrong} of them")
print(f"Questions where all 5 agree : {len(unanimous)}, and the single call was wrong on {unanimous_wrong} of them")

# ==========================================================================
# Step 6: Store the result for the final table
# ==========================================================================
record_result(
    task="15 fee calculations", technique="self-consistency",
    result=f"{vote_right} of {len(votes)} right", accuracy=vote_accuracy,
    calls=VOTES, cost=vote_cost / len(votes),
    seconds=vote_seconds_in_a_row, seconds_parallel=vote_seconds_in_parallel,
)

# ==========================================================================
# Step 7: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ You have 75 answers: 5 for each of the 15 calculations" if len(all_answers) == 75 and all(
      len(answers) == 5 and all(answer["id"] == calculation["id"] for answer in answers)
      for calculation, answers in zip(CALCULATIONS, answers_per_question))
      else f"❌ Expected 5 answers for each of 15 calculations, got {len(all_answers)} answers in all")
votes_are_most_common = all(
    vote["winner"] is None or vote["amounts"].count(vote["winner"]) == max(
        vote["amounts"].count(amount) for amount in vote["amounts"] if amount is not None)
    for vote in votes)
print("✅ Every vote is the most common of its five answers" if votes_are_most_common and all(
      vote["agreement"] == vote["amounts"].count(vote["winner"]) for vote in votes if vote["winner"] is not None)
      else "❌ A vote is not the most common of its five answers: check majority_vote in Step 2")
print("✅ The vote used exactly 5 times the calls of the baseline" if len(vote_calls) == 5 * len(baseline_calls)
      else f"❌ Expected {5 * len(baseline_calls)} calls, counted {len(vote_calls)}")
print("✅ The self-consistency row is stored in RESULTS" if any(
      row["technique"] == "self-consistency" for row in RESULTS) else "❌ The self-consistency row is missing in RESULTS")
if not split:
    print("🤔 All five answers agreed on every question in your run. A vote cannot change an answer that never "
          "varies: you paid for five calls and got the same result. Check that VOTE_TEMPERATURE is 0.7, not 0.")
elif vote_right > BASELINE["right"]:
    print(f"🤔 The vote got {vote_right} of 15 right, the single call {BASELINE['right']}. In your run, five times "
          f"the calls ({times_as_much(vote_cost, BASELINE['cost'])} the cost) bought {vote_right - BASELINE['right']} "
          "more right answer(s).")
elif vote_right == BASELINE["right"]:
    print(f"🤔 The vote and the single call both got {vote_right} of 15 right in your run. With 15 questions, "
          "a difference of one answer can come and go between runs.")
else:
    print(f"🤔 The vote got {vote_right} of 15 right, fewer than the single call ({BASELINE['right']}). "
          "The single call was probably lucky on a question the model usually gets wrong. Run both cells again to see.")
if unanimous_wrong:
    print(f"🤔 The single call was also wrong on {unanimous_wrong} question(s) where all five answers agree. Look at "
          "the vote there. If the vote is right, the single call slipped once. If the vote is wrong too, the model "
          "makes the same mistake every time, and no number of calls will fix it.")
print("🧠 A vote removes random slips for five times the calls. It cannot remove a mistake the model makes every time.")

# ⛓️ Topic 3: Prompt Chaining, or One Small Task per Call

Some jobs are really several jobs in a row. Take this message from a bank customer:

> *"I got back from Lisbon yesterday and my statement is a mess. Every time I took cash out of a machine over there I was charged an extra fee, and nobody told me how it is worked out. On top of that, my card stopped giving me cash on Friday after two withdrawals, as if there were some cap on what I can take out in a day. And why is there a separate fee line under each restaurant bill I paid by card? I'm on the Standard plan."*

To write a good reply, somebody has to do three things: notice that there are three separate issues, look up what the fee policy says about each one, and write a friendly reply that covers all of them. When you ask for the reply in one call, the model does all three in one go, and you see only the final text. If the reply skips the restaurant fee, you cannot tell whether the model missed the issue, did not find the policy, or dropped it while writing.

**Prompt chaining** splits the job into several calls. Each call has **one small task**, and **the output of one call is the input of the next**:

| Step | The call's one task | Its output for the message above |
|---|---|---|
| 1 | List the issues in the message | 1. fee for cash withdrawals abroad · 2. a cap on cash per day · 3. a fee line under each card payment abroad |
| 2 | Find the policy for each issue | 1. section 3: \$5 plus 1.5% of the amount · 2. section 3: \$500 per day · 3. section 4: 2% on the Standard plan |
| 3 | Write the reply from steps 1 and 2 | *"Thank you for your message. Each cash withdrawal abroad costs \$5 plus 1.5% of the amount ..."* |

The outputs between the steps are called **intermediate results**. They are the main thing a chain gives you, for two reasons:

- **Each call has one task.** The first call only looks for issues. The second only looks things up. The last call writes from a checklist and does not have to find anything.
- **Every step can be read and checked separately.** When a reply is wrong, you read three short texts and see which step failed. You can also check a step with code before going on, for example "step 1 must list at least one issue".

A chain has costs too. Three calls cost more than one. Each call needs the output of the one before it, so they **cannot run at the same time**: the waiting times add up. And a mistake travels down the chain: an issue that step 1 misses can never appear in the reply.

So where does a chain pay? When the job is too big for one prompt, when the steps need different inputs or tools (one step searches a database, the next one writes), or when you want to check a step with code before going on. On a short message with a three-page policy, one call can hold the whole job, and a chain mainly adds calls and waiting time.

---

### 🔍 What You'll Learn

- Split one job into a chain of calls, where each call gets the output of the previous one
- Read intermediate results to find the step where something went wrong
- Check each step of a chain with code, and add up what the chain costs in calls, money and waiting time

### 🧠 Why This Matters

- Real jobs often have steps that cannot share one prompt: one step needs a search, the next a calculation, the last a reply. A chain is how those steps are put together
- When a single call fails you can only change the prompt and hope. When a chain fails you know which step to fix

## 🧪 Exercise 3: Split the Reply to a Complaint Into a Chain of Three Calls

You work with 20 customer complaints. Each one contains **three different issues**, and each issue is answered by one fact of the bank's fee policy (a fee, a limit or a deadline). A provided function, `issues_dealt_with`, checks a text with code: an issue counts as dealt with when the text states the right policy fact for it.

You build a **chain of three calls**: the first lists the issues, the second finds the policy for each issue, the third writes the reply from those two results. You run it on all 20 complaints. Then you look at what a chain gives you and what it costs:

- **Each step can be checked.** The checker runs on the output of step 2 and on the final reply. So for every issue you can see whether its policy fact was found, and whether it made it into the reply.
- **Each step can be read.** For one complaint you print the three intermediate results and find the step where an issue was lost.
- **The cost adds up.** The three calls must run one after another, so the waiting time of a reply is the sum of three calls. You print the time and the cost of each step.

This exercise does not set the chain against a single call. On a short message with a three-page policy, one call can hold the whole job, so a chain is not expected to write better replies here. It is the right tool when a job is too big for one prompt, when steps need different inputs or tools, or when you want to check a step with code before going on. Here you build one and learn to read it.

### 🎯 What You'll Do

1. Load the complaints and the fee policy, and read one complaint with its three issues.
2. Write the three prompts of the chain and pass each output on to the next call.
3. Answer all 20 complaints with the chain.
4. Check step 2 and the final reply with code, for all 60 issues.
5. Add up what a reply costs: calls, dollars and seconds, step by step.
6. Print the intermediate results of one complaint and find the step where an issue was lost.

👉 By the end, you'll have a working three-step chain, and you'll know how to check it step by step and what it costs.

---

### ✅ You Succeed When

- You have 20 chain replies
- Every chain made exactly 3 calls, and each call after the first received the output of the call before it
- The chain row is stored in `RESULTS`

---

### 🛠 Technical Info

- **The checker:** `issues_dealt_with(text, complaint)` returns three `True`/`False` values, one per issue. It works on any text: the output of step 2 or the final reply
- **Passing a result on:** the reply of a call is `call["text"]`; you put it into the next prompt with an f-string
- **Honest note about the check:** the checker looks for the policy fact in the reply (for example "\$5" and "1.5%"), with patterns. It cannot judge tone, and a reply that states the fact in unusual words can be missed.
- **Honest note about the time:** the notebook works on up to 8 complaints at the same time, but inside one chain the three calls run one after another. The chain's response time is the sum of its three measured calls.
- **Honest note about the result:** the count of replies that deal with all three issues is information about this chain, not a score to beat. In our runs it was 14 to 18 of 20 with this small model, and most of the issues that went missing were lost in step 3.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Add a check between step 1 and step 2: if step 1 listed fewer than two issues, run step 1 again. Count how many chains needed it.
2. Write the single call yourself: one prompt with the policy and the message that asks for the reply. Run it on the 20 complaints and compare its count, cost and time with the chain's. Repeat both a few times before you conclude anything.

In [ ]:
# ==========================================================================
# ⛓️ Exercise 3: Split the Reply to a Complaint Into a Chain of Three Calls
# ==========================================================================

# Make sure the Setup cell ran first (it creates call_model and the other helpers)
assert "call_model" in globals(), "Run the Setup cell at the top of the notebook first."

# ==========================================================================
# Step 1: Load the complaints and the fee policy (upload the files if they are not here yet)
# ==========================================================================
COMPLAINTS_FILE = "customer_complaints.json"
POLICY_FILE = "bank_fee_policy.md"
for name in (COMPLAINTS_FILE, POLICY_FILE):
    if not file_is_ready(name):
        print(f"📁 Please upload '{name}':")
        files.upload()
    assert file_is_ready(name), f"'{name}' is missing or does not open. Upload it again (the exact name matters)."
    print(f"📋 {name}: {describe_file(name)}")

with open(COMPLAINTS_FILE) as f:
    COMPLAINTS = json.load(f)    # a list of 20 dictionaries: id, tone, message, issues
with open(POLICY_FILE) as f:
    POLICY = f.read()            # the fee policy as one text, about 1,450 words

# ==========================================================================
# Step 2: Look at one complaint and its three issues
# ==========================================================================
print()
print("=" * 70)
print("📊 ONE COMPLAINT AND ITS THREE ISSUES")
print("=" * 70)
example = COMPLAINTS[0]
print(f"{example['id']} (tone: {example['tone']})")
print(textwrap.fill(example["message"], width=100))
for number, issue in enumerate(example["issues"], start=1):
    print(f"\n   Issue {number}: {issue['issue']}")
    print(f"   Policy fact (section {issue['section']}): {issue['fact']}")


# ==========================================================================
# Step 3: The checker (provided): which issues does a reply deal with?
# ==========================================================================
# Each issue has a list of patterns in "check" (regular expressions, for example one that finds "$5").
# A text deals with an issue when EVERY pattern of that issue is found in it.
def issues_dealt_with(text, complaint):
    """Return three True/False values: for each issue of the complaint, does the text state its policy fact?"""
    plain_text = text.replace("*", "")   # models often write numbers in **bold**; the stars would break a pattern
    return [
        all(re.search(pattern, plain_text, re.IGNORECASE) for pattern in issue["check"])   # IGNORECASE: "Free" = "free"
        for issue in complaint["issues"]
    ]


REPLY_MAX_TOKENS = 1500   # room for a list of issues, a list of policy lines or a reply of about 150 words


# ==========================================================================
# Step 4: The chain: three calls, each with one small task
# ==========================================================================
# Call 1 lists the issues. Call 2 gets that list and finds the policy for each issue.
# Call 3 gets the list and the policy lines and writes the reply. Each call needs the output of the one before.
def chain_reply(complaint):
    """Answer one complaint with a chain of three calls. Return the reply, the check, the calls and the steps."""
    # Call 1: only find the issues. It does not see the policy and must not answer anything.
    # TODO 1: Write the prompt for call 1 (an f-string, 3 to 5 sentences). It must contain the customer's
    # message, complaint['message'], and ask for one thing only: a numbered list of the separate issues
    # in the message, also the ones mentioned in passing, each written as a full question that can be
    # looked up without the message. Tell the model not to answer the issues.
    list_prompt = ___  # <-- Fill in the blank (several lines)
    first = call_model(list_prompt, max_tokens=REPLY_MAX_TOKENS)
    # TODO 2: Keep the reply of call 1 as the intermediate result. Hint: first["text"]
    issues = ___  # <-- Fill in the blank

    # Call 2: only look things up. Its input is the output of call 1 (plus the message, for the details).
    policy_prompt = (
        f"<fee_policy>\n{POLICY}\n</fee_policy>\n\n"
        f"<customer_message>\n{complaint['message']}\n</customer_message>\n\n"
        # TODO 3: Put the output of call 1 between the <issues> tags
        f"<issues>\n{___}\n</issues>\n\n"  # <-- Fill in the blank
        "For each issue in the list, find the part of the fee policy that answers it. Write one line per issue: "
        "the issue number, the section number, and the sentence or sentences of that section that answer it, "
        "copied word for word, with every fee, limit or deadline in them. Look through all 12 sections before "
        "you write 'not in the policy'."
    )
    second = call_model(policy_prompt, max_tokens=REPLY_MAX_TOKENS)
    # TODO 4: Keep the reply of call 2 as the intermediate result
    policies = ___  # <-- Fill in the blank

    # Call 3: only write. Its input is the message plus the outputs of calls 1 and 2 (not the whole policy).
    write_prompt = (
        "You answer customers for a bank's support team.\n\n"
        f"<customer_message>\n{complaint['message']}\n</customer_message>\n\n"
        # TODO 5: Call 3 gets both intermediate results: the list of issues and the policy found for each
        f"<issues>\n{___}\n</issues>\n\n"  # <-- Fill in the blank
        f"<policy_for_each_issue>\n{___}\n</policy_for_each_issue>\n\n"  # <-- Fill in the blank
        "Write the reply to this customer. Go through the list of issues in order and answer every one of them, "
        "with the exact fee, limit or deadline found for it. Do not leave an issue out to save space. "
        "Write to the customer directly, in a friendly and clear tone, in at most 150 words."
    )
    third = call_model(write_prompt, max_tokens=REPLY_MAX_TOKENS)

    return {"id": complaint["id"], "reply": third["text"], "dealt": issues_dealt_with(third["text"], complaint),
            "calls": [first, second, third], "issues": issues, "policies": policies,
            "prompts": [list_prompt, policy_prompt, write_prompt]}


chain_replies = run_in_parallel(chain_reply, COMPLAINTS)

# ==========================================================================
# Step 5: Check each step with code
# ==========================================================================
# The checker runs on two texts of every chain: the output of step 2 (was the policy fact found?) and the
# final reply (did the fact make it into the reply?). An issue can be lost before step 3 or in step 3.
print()
print("=" * 70)
print("📊 ISSUES AFTER STEP 2 AND IN THE REPLY, PER COMPLAINT")
print("=" * 70)
print(f"{'id':<6}{'fact found by step 2':<24}{'fact in the reply':<20}")
for complaint, chain in zip(COMPLAINTS, chain_replies):
    chain["found_by_step_two"] = issues_dealt_with(chain["policies"], complaint)
    print(f"{chain['id']:<6}{str(sum(chain['found_by_step_two'])) + ' of 3':<24}{str(sum(chain['dealt'])) + ' of 3':<20}")

issue_count = 3 * len(COMPLAINTS)
found_by_step_two = sum(sum(chain["found_by_step_two"]) for chain in chain_replies)   # of all 60 issues
in_the_reply = sum(sum(chain["dealt"]) for chain in chain_replies)
lost_in_step_three = sum(found and not dealt for chain in chain_replies
                         for found, dealt in zip(chain["found_by_step_two"], chain["dealt"]))
# TODO 6: Count the replies that deal with all three issues. Hint: all(reply["dealt"]) is True only when
# all three values are True, and sum() counts the True values of a list
chain_complete = ___  # <-- Fill in the blank
print(f"\nPolicy fact found by step 2            : {found_by_step_two} of {issue_count} issues")
print(f"Policy fact in the final reply         : {in_the_reply} of {issue_count} issues")
print(f"Found by step 2, missing in the reply  : {lost_in_step_three} (lost in step 3)")
print(f"Replies that deal with all 3 issues    : {chain_complete} of {len(COMPLAINTS)}")

# ==========================================================================
# Step 6: What a reply costs: three calls, one after another
# ==========================================================================
# Each call of a chain needs the output of the one before, so the waiting times ADD UP.
# We print the average time and cost of each step, and the sum for one reply.
chain_calls = [call for reply in chain_replies for call in reply["calls"]]
print()
print("=" * 70)
print("📊 THE COST OF ONE REPLY, STEP BY STEP")
print("=" * 70)
print(f"{'':<36}{'seconds':>10}{'cost':>12}")
for step, name in enumerate(["Step 1: list the issues", "Step 2: find the policy", "Step 3: write the reply"]):
    step_calls = [reply["calls"][step] for reply in chain_replies]   # this step's call of every chain
    print(f"{name:<36}{sum(call['seconds'] for call in step_calls) / len(step_calls):>10.1f}"
          f"{'$' + format(sum(call['cost'] for call in step_calls) / len(step_calls), '.5f'):>12}")
chain_cost = sum(call["cost"] for call in chain_calls) / len(COMPLAINTS)
chain_seconds = sum(call["seconds"] for call in chain_calls) / len(COMPLAINTS)
print(f"{'One reply: 3 calls in a row':<36}{chain_seconds:>10.1f}{'$' + format(chain_cost, '.5f'):>12}")
show_bill(chain_calls, "Exercise 3")

# ==========================================================================
# Step 7: Read the intermediate results of one complaint
# ==========================================================================
# We show a complaint whose reply misses an issue, because that is when reading the steps pays.
# If every reply deals with all three issues in this run, we show the first complaint.
incomplete = [index for index in range(len(COMPLAINTS)) if not all(chain_replies[index]["dealt"])]
shown = (incomplete or [0])[0]
complaint, chain = COMPLAINTS[shown], chain_replies[shown]

print()
print("=" * 70)
print(f"🔍 INSIDE THE CHAIN FOR {complaint['id']}")
print("=" * 70)
print("THE MESSAGE:")
print(textwrap.fill(complaint["message"], width=100))
print("\nSTEP 1 (the issues):")
# Each line of the model's list is wrapped at 100 characters, so long lines stay readable
print("\n".join(textwrap.fill(line, width=100, subsequent_indent="   ") for line in chain["issues"].split("\n")))
print("\nSTEP 2 (the policy for each issue):")
print("\n".join(textwrap.fill(line, width=100, subsequent_indent="   ") for line in chain["policies"].split("\n")))
print("\nSTEP 3 (the reply):")
print(textwrap.fill(chain["reply"], width=100))

# For each of the three issues: where is its policy fact, according to the checker?
print()
for issue, found, dealt in zip(complaint["issues"], chain["found_by_step_two"], chain["dealt"]):
    if dealt:
        verdict = "in the reply"
    elif found:
        verdict = "LOST IN STEP 3: step 2 found the fact, the reply does not state it"
    else:
        verdict = "LOST BEFORE STEP 3: read step 1 (is the issue in the list?) and step 2 (which policy did it pick?)"
    print(f"'{issue['issue']}'")
    print(f"   fact: {issue['fact']}")
    print(f"   {verdict}")

# ==========================================================================
# Step 8: Store the result for the final table
# ==========================================================================
record_result(
    task="20 complaints with 3 issues", technique="prompt chaining",
    result=f"{chain_complete} of {len(COMPLAINTS)} replies deal with all 3", accuracy=chain_complete / len(COMPLAINTS),
    calls=3, cost=chain_cost, seconds=chain_seconds,
)

# ==========================================================================
# Step 9: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ You have 20 chain replies" if len(chain_replies) == 20
      else f"❌ Expected 20 chain replies, got {len(chain_replies)}")
three_calls_each = all(len(reply["calls"]) == 3 for reply in chain_replies)
# Each later prompt must contain the output of the call before it (only checked where that output is not empty)
passed_on = all(
    (not reply["issues"] or (reply["issues"] in reply["prompts"][1] and reply["issues"] in reply["prompts"][2]))
    and (not reply["policies"] or reply["policies"] in reply["prompts"][2])
    for reply in chain_replies)
print("✅ Every chain made exactly 3 calls, and each call after the first received the output of the call before it"
      if three_calls_each and passed_on
      else "❌ A chain did not make 3 calls, or a prompt does not contain the previous output: check Step 4")
print("✅ The chain row is stored in RESULTS" if any(
      row["technique"] == "prompt chaining" for row in RESULTS) else "❌ The chain row is missing in RESULTS")
print(f"🤔 In your run {chain_complete} of 20 replies deal with all three issues, and {lost_in_step_three} issue(s) "
      f"were found by step 2 and lost in step 3. Each reply took 3 calls and {chain_seconds:.1f} s of waiting.")
print("🧠 A chain gives every call one small task and lets you read and check each step. It pays with calls that must wait for each other.")

# 🪞 Topic 4: Reflection, or a Second Call That Reviews the First Answer

People check their work before handing it in. **Reflection** gives a model the same second look: after the first call has answered, a **second call** reads the question and that answer, and corrects the answer if it finds a mistake.

Here is the instant-transfer question from Topic 1 again (correct total: \$14.50), with a first answer that forgot both limits:

| | What the call gets | What comes back |
|---|---|---|
| First call | the fee table and the question | `ANSWER: 29.30` |
| Review call | the fee table, the question, the first answer, and the instruction "Recalculate each step and compare it with the fee table" | *\$30: 1% is \$0.30, the table says minimum \$0.50, so \$0.50. \$400: \$4.00. \$2,500: 1% is \$25, the table says maximum \$10, so \$10.00.* `ANSWER: 14.50` |

The review found the mistake because its instruction made it do something the first call had not done: go through the calculation line by line against the table.

That is the part to remember. A review is only as good as **what the reviewer has to check against**. Compare two review instructions:

- **A bare review:** *"Review your answer and correct it if it is wrong."* The reviewer has the same table, the same question and the same model as the first call. Nothing new has entered. It is a second attempt with a hint that something may be wrong.
- **A review with something to check against:** a checklist, the source document, the output of a test, the result of a tool. Now the reviewer can compare the answer with something, step by step.

A bare review has a known risk. The instruction "correct it if it is wrong" suggests that there is something to correct. A model that was right the first time can talk itself into a "correction" and **turn a right answer into a wrong one**. So a review can move an answer in two directions: from wrong to right, and from right to wrong. A review that fixes two answers and breaks two has cost you a second call for nothing.

The price of reflection is one more call per answer. The review needs the first answer, so the two calls run one after another, and the waiting times add up.

---

### 🔍 What You'll Learn

- Add a review call that receives the question and the first answer
- Give the reviewer something to check against, and see how that differs from a bare "review your answer"
- Count a review's changes in both directions: wrong to right, and right to wrong

### 🧠 Why This Matters

- "Have the model check its own answer" sounds safe, and a bare review can quietly lower your accuracy while doubling your cost
- The useful version carries over to real systems: a review step that gets test results, a source document or a checklist is how coding agents and document checkers catch their own mistakes

## 🧪 Exercise 4: Add a Second Call That Reviews the First Answer

You take the 15 fee calculations and the answers the single call gave in Exercise 1. Those first answers are the base version, and they are not asked again. For each one you add a **second call** that reviews it, in two versions:

- **Version 1, the bare review:** *"Review your answer and correct it if it is wrong."*
- **Version 2, the review with the fee table:** *"Recalculate each step and compare it with the fee table."* The reviewer has to copy the row of the fee table that applies to each fee, check the conditions in that row, and work the amount out again.

For each version you count two things: wrong answers that became right, and right answers that became wrong. Then you compare the final accuracy of both versions with the baseline, next to what the second call cost in money and waiting time. One answer now takes two calls, one after the other.

Both reviewers get the same material: the fee table, the question and the first reply with its working. Only the instruction differs. Do not expect a fixed outcome from the bare review: a review without anything to check against may fix some answers, may break some, and may do both. The numbers of your run say which.

### 🎯 What You'll Do

1. Write the two review instructions.
2. Write the review call: it gets the fee table, the question, the first answer and one instruction.
3. Run both versions on the 15 first answers from Exercise 1 (30 calls).
4. Count wrong-to-right and right-to-wrong for each version.
5. Compare the final accuracy, the cost and the response time with the baseline, and store both rows.

👉 By the end, you'll know what each kind of review did to the 15 first answers, in both directions.

---

### ✅ You Succeed When

- You have 15 reviewed answers for each of the two versions
- For each version, the counts add up: right after the review = right before + wrong-to-right - right-to-wrong
- Both reflection rows are stored in `RESULTS`

---

### 🛠 Technical Info

- **The first answers:** `baseline_answers` from Exercise 1; each has `amount`, `right` and the whole reply in `call["text"]`
- **The two directions:** wrong to right means `not first["right"] and reviewed["right"]`; right to wrong is the opposite
- **Honest note:** both reviewers see the fee table, because the question cannot be judged without it. The difference is the instruction: version 2 tells the reviewer to copy each row of the table and to redo every step against it. In our seven runs the bare review ended between 1 answer below and 3 above the first answers and broke up to 3 right answers. The review with the fee table ended 3 to 8 answers above and broke 0 or 1. With 15 questions, a difference of one answer can be chance, so look at the two directions and not only at the total.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Write a third review instruction of your own that gives the reviewer a short checklist ("check every minimum, every maximum, every 'first N are free'") and compare it with version 2.
2. Run the review of version 2 only on the questions where the vote in Exercise 2 was split, and count how many calls that saves.

In [ ]:
# ==========================================================================
# 🪞 Exercise 4: Add a Second Call That Reviews the First Answer
# ==========================================================================

# Make sure the Setup cell and Exercise 1 ran first (Exercise 1 creates the first answers)
assert "call_model" in globals(), "Run the Setup cell at the top of the notebook first."
assert "BASELINE" in globals(), "Run Exercise 1 first: this exercise reviews its answers."

# ==========================================================================
# Step 1: The two review instructions
# ==========================================================================
# Version 1 gives the reviewer nothing to check against. Version 2 tells it to redo each step against the fee table.
BARE_REVIEW = "Review your answer and correct it if it is wrong."
# TODO 1: Write the instruction of version 2 (a string, 3 to 5 sentences). Start from "Recalculate each step
# and compare it with the fee table, one fee at a time." and say what the reviewer must do for each fee:
# copy the row of the fee table that applies, check the conditions in it (a minimum, a maximum, a free
# allowance, a limit), work out the amount. Do not describe the ANSWER line: ANSWER_FORM below adds it.
TABLE_REVIEW = ___  # <-- Fill in the blank (several lines)

# Each instruction is followed by the form of the answer, so that read_amount can find the number.
ANSWER_FORM = ("Finish with one last line in exactly this form (the total in US dollars, with two decimals):\n"
               "ANSWER: 12.50")
REVIEW_INSTRUCTIONS = {
    "reflection, bare review": BARE_REVIEW + "\n\n" + ANSWER_FORM,
    "reflection, review with the fee table": TABLE_REVIEW + "\n\n" + ANSWER_FORM,
}


# ==========================================================================
# Step 2: The review call
# ==========================================================================
# The second call gets everything the first call had (fee table and question), plus the first reply
# with its working, and one of the two instructions.
def review(calculation, first_answer, instruction):
    """One review call for one first answer. Return the reviewed amount, whether it is right, and the call."""
    first_reply = first_answer["call"]["text"]   # the whole first reply: the working and the ANSWER line
    prompt = (
        "You work in a bank's support team. Earlier you answered the question below with the help of the fee table.\n\n"
        f"<fee_table>\n{FEE_TABLE}</fee_table>\n\n"
        f"<question>\n{calculation['question']}\n</question>\n\n"
        # TODO 2: Show the reviewer the first reply (first_reply), then give it the review instruction
        f"<your_first_answer>\n{___}\n</your_first_answer>\n\n"  # <-- Fill in the blank
        f"{___}"  # <-- Fill in the blank
    )
    call = call_model(prompt, max_tokens=ANSWER_MAX_TOKENS)
    amount = read_amount(call["text"])
    return {"id": calculation["id"], "amount": amount, "right": is_right(amount, calculation["answer"]), "call": call}


# ==========================================================================
# Step 3: Run both versions on the 15 first answers
# ==========================================================================
# pairs: each calculation together with its first answer from Exercise 1.
pairs = list(zip(CALCULATIONS, baseline_answers))
reviews = {}   # version name: the 15 reviewed answers
for version, instruction in REVIEW_INSTRUCTIONS.items():
    # lambda pair: ... is a small function without a name; it reviews one (calculation, first answer) pair
    reviews[version] = run_in_parallel(lambda pair: review(pair[0], pair[1], instruction), pairs)

# ==========================================================================
# Step 4: Count the changes in both directions
# ==========================================================================
# wrong to right: the review fixed the answer. right to wrong: the review broke a correct answer.
review_counts = {}
for version, reviewed_answers in reviews.items():
    wrong_to_right = [reviewed["id"] for first, reviewed in zip(baseline_answers, reviewed_answers)
                      # TODO 3: wrong to right: the first answer was wrong and the reviewed answer is right
                      if ___ and ___]  # <-- Fill in the blanks
    right_to_wrong = [reviewed["id"] for first, reviewed in zip(baseline_answers, reviewed_answers)
                      # TODO 4: right to wrong: the first answer was right and the reviewed answer is wrong
                      if ___ and ___]  # <-- Fill in the blanks
    review_counts[version] = {
        "wrong_to_right": wrong_to_right,
        "right_to_wrong": right_to_wrong,
        "right": sum(reviewed["right"] for reviewed in reviewed_answers),
        "cost": sum(reviewed["call"]["cost"] for reviewed in reviewed_answers),
        "seconds": sum(reviewed["call"]["seconds"] for reviewed in reviewed_answers) / len(reviewed_answers),
    }

print("=" * 70)
print("📊 EVERY ANSWER BEFORE AND AFTER THE REVIEW")
print("=" * 70)
print(f"{'id':<5}{'correct':>9}{'first answer':>15}{'bare review':>15}{'with fee table':>17}")
for index, calculation in enumerate(CALCULATIONS):
    cells = []
    for answers in (baseline_answers, *reviews.values()):   # the first answers, then each version's reviews
        answer = answers[index]
        cells.append(f"{as_dollars(answer['amount'])} {'ok' if answer['right'] else 'x '}")
    print(f"{calculation['id']:<5}{as_dollars(calculation['answer']):>9}{cells[0]:>15}{cells[1]:>15}{cells[2]:>17}")

# ==========================================================================
# Step 5: Compare with the baseline
# ==========================================================================
# One reviewed answer costs the first call plus the review call, and the customer waits for both.
print()
print("=" * 70)
print("📊 WHAT EACH REVIEW DID")
print("=" * 70)
first_cost = BASELINE["cost"] / len(CALCULATIONS)   # the cost of one first answer
print(f"Single call (no review): {BASELINE['right']} of 15 right · 1 call · ${first_cost:.5f} and "
      f"{BASELINE['seconds']:.1f} s per answer")
for version, counts in review_counts.items():
    cost_per_answer = first_cost + counts["cost"] / len(CALCULATIONS)
    # TODO 5: The customer waits for the first call and then for the review. Hint: BASELINE["seconds"] is the
    # average time of a first call, counts["seconds"] the average time of a review call
    seconds_per_answer = ___  # <-- Fill in the blank
    print(f"\n{version}:")
    print(f"   wrong answers that became right : {len(counts['wrong_to_right'])}  ({', '.join(counts['wrong_to_right']) or 'none'})")
    print(f"   right answers that became wrong : {len(counts['right_to_wrong'])}  ({', '.join(counts['right_to_wrong']) or 'none'})")
    print(f"   right after the review          : {counts['right']} of 15 (before: {BASELINE['right']})")
    print(f"   per answer                      : 2 calls · ${cost_per_answer:.5f} ({times_as_much(cost_per_answer, first_cost)} "
          f"the single call) · {seconds_per_answer:.1f} s ({times_as_much(seconds_per_answer, BASELINE['seconds'])})")
    record_result(
        task="15 fee calculations", technique=version,
        result=f"{counts['right']} of 15 right", accuracy=counts["right"] / len(CALCULATIONS),
        calls=2, cost=cost_per_answer, seconds=seconds_per_answer,
    )
print()
show_bill([reviewed["call"] for answers in reviews.values() for reviewed in answers], "Exercise 4")

# ==========================================================================
# Step 6: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ You have 15 reviewed answers for each of the two versions"
      if len(reviews) == 2 and all(len(answers) == 15 for answers in reviews.values())
      else "❌ Expected 2 versions with 15 reviewed answers each")
counts_add_up = all(
    counts["right"] == BASELINE["right"] + len(counts["wrong_to_right"]) - len(counts["right_to_wrong"])
    for counts in review_counts.values())
print("✅ For each version, the counts add up: right after = right before + wrong-to-right - right-to-wrong"
      if counts_add_up else "❌ The counts do not add up: check the two conditions in Step 4")
print("✅ Both reflection rows are stored in RESULTS" if sum(
      row["technique"].startswith("reflection") for row in RESULTS) == 2
      else "❌ A reflection row is missing in RESULTS")
bare, with_table = (review_counts[version] for version in REVIEW_INSTRUCTIONS)
if bare["right_to_wrong"]:
    print(f"🤔 The bare review turned {len(bare['right_to_wrong'])} right answer(s) into wrong ones in your run. "
          "That is the risk of a review with nothing to check against.")
else:
    print("🤔 The bare review broke no right answer in your run. It can happen in another run: the reviewer had "
          "no new information, only a hint that something may be wrong.")
if with_table["right"] > bare["right"]:
    print(f"🤔 The review with the fee table ended at {with_table['right']} of 15, the bare review at {bare['right']}. "
          "Giving the reviewer something to check against made the second call worth more.")
elif with_table["right"] == bare["right"]:
    print(f"🤔 Both reviews ended at {bare['right']} of 15 in your run. Compare the two directions above: "
          "the totals can be equal while one version breaks answers and the other does not.")
else:
    print(f"🤔 In your run the bare review ended higher ({bare['right']} of 15) than the review with the fee table "
          f"({with_table['right']}). With 15 questions, one or two answers can be chance. Run the cell again to see.")
if BASELINE["right"] == len(CALCULATIONS):
    print("🤔 The first answers were all right, so a review had nothing to fix and could only break answers or leave them.")
print("🧠 Count a review in both directions. A second look helps when the reviewer has something to check against.")

# 🧩 Topic 5: Divide-and-Conquer Planning, or Letting the Model Split a Long Job

Some jobs are long. Imagine a file with 200 bank statements, about 3,500 charges, and the job: *find every charge that was taken twice.* One way is to hand the whole file to a model in one call. A current model with room to think can do that: it goes through the statements one by one in its thinking and writes the list at the end. But the whole job then hangs on one long call. You wait minutes for it, you see nothing until it ends, and if it fails or runs into its output limit, all the work is lost. And a file fifty times this size no longer fits into one call at all.

**Divide-and-conquer planning** handles a job like this in three stages:

1. **Plan.** One call gets a description of the job, not the data, and writes a **plan**: how to split the job into parts, what each part has to do, and how to put the results together. The prompt that asks for this is called a **planner prompt**. Its output is not an answer. It is instructions for the calls that follow.
2. **Solve each part.** Your code runs one call per part, each with a small piece of the data and the instructions from the plan.
3. **Combine.** One last call gets all the findings and writes the final answer.

For the 200 statements, a plan can look like this:

```text
GROUP_SIZE: 8
GROUP_TASK: Check each statement by itself. Flag a charge only when an earlier charge in the same
            statement has the same merchant, amount and date. Write one line per finding.
COMBINE_TASK: Collect the findings of all groups and remove any that do not meet the rule.
```

Your code reads `GROUP_SIZE: 8`, cuts the 200 statements into 25 groups, and sends 25 calls that each look at about 140 charges instead of 3,500.

Three things decide whether planning is worth it:

- **A checkable goal.** "Find the duplicate charges" has a clear test for every finding: same merchant, same amount, same date. The plan can state it, and each part can apply it. A goal like "make the reply nicer" gives a plan nothing to aim at.
- **A step budget.** A plan decides how many calls will be made, so it also decides the cost. You give the planner a limit ("at most 30 calls") and your **code enforces it**: if the plan asks for more, the code makes the groups larger. Never trust a plan to keep its own budget.
- **The size of the job.** Planning adds a call before the work and a call after it. On a job with 3,500 lines that is a small extra. On one customer question it triples the calls and the waiting time, and there is nothing to divide. The same planner prompt that suits a long job is wasted on a single chat reply.

Because the parts do not depend on each other, the calls for the groups can run at the same time. Only the plan has to come before them and the combine call after them.

So where does a plan pay? In four situations: the job is larger than what the model can read in one call or write in one answer; one failed call must not lose all the work, because a failed part can be run again by itself; the job has to finish sooner than one long call would, because the parts run at the same time; or different parts need different tools or instructions. When one call can do the whole job and nobody is in a hurry, a plan adds calls and nothing else.

---

### 🔍 What You'll Learn

- Write a planner prompt whose output is a plan that code can read
- Run the plan as code: split the job into groups, one call per group, one call to combine
- Enforce a step budget in code, and check the result against a goal that can be tested
- Tell a job that deserves a plan from one that does not

### 🧠 Why This Matters

- A long job in one call is all or nothing: if the call fails or is cut off at its output limit, you have no result and cannot see how far it got
- A planner without a budget can turn one job into hundreds of calls. The limit belongs in your code, not in the prompt

## 🧪 Exercise 5: Have the Model Plan a Long Job and Solve It in Parts

The job: find the duplicate charges in a file of 200 bank statements with about 3,500 lines. A **duplicate charge** is a second charge in the same statement with exactly the same merchant name, the same amount and the same date as an earlier one. 56 duplicates were planted in the file and their ids are stored, so the result can be checked. The two identical lines of a planted duplicate are never next to each other. The file also contains look-alikes that are **not** duplicates: the same subscription a month apart, the same fare on two days in a row, two amounts a few cents apart, two merchants with almost the same name, a charge and its refund. So you measure two things: how many of the planted duplicates were found, and how many **false alarms** were raised (charges reported as duplicates that are not).

First you give the **whole file to the model in one call**, at a fair setting: it may think before it answers, and it has enough room to write. Then you use a **planner prompt**: the model writes a plan that splits the statements into groups, your code runs one call per group with a **limit of 30 calls** in total, and a last call combines the findings. You put both results side by side as they are: duplicates found, false alarms, calls, cost and elapsed seconds.

Do not expect the planner to find more. In our runs the single call found 55 or 56 of the 56: a current model with room to think works through the statements one by one in its thinking. What the plan changes is something else, and the output shows it: the waiting time (the group calls run at the same time), that you can read what each group found, and that the number of calls is limited by your code.

Then you turn the comparison around. You use the **same planner** on one simple customer question and compare it with a single call. Here the plan is expected to add cost and waiting time and no accuracy.

### 🎯 What You'll Do

1. Load the statements, read one, and turn all of them into the text the model reads.
2. Give the whole file to the model in one call and score what it finds.
3. Write `plan_and_solve`: one planning call, a budget check in code, one call per group, one combining call.
4. Run it on the 200 statements and compare it with the single call.
5. Run the same planner on one simple question and compare it with a single call.

👉 By the end, you'll have a reusable planner with a call budget, and real numbers for a long job and for a job that is too small for a plan.

---

### ✅ You Succeed When

- The groups of the plan cover all 200 statements, each exactly once
- The planner run on the statements used at most 30 calls: 1 for the plan, 1 per group, 1 to combine
- On the simple question, the planner used more calls than the single call
- The four rows of this exercise are stored in `RESULTS`

---

### 🛠 Technical Info

- **The plan's format:** three lines, `GROUP_SIZE:`, `GROUP_TASK:` and `COMBINE_TASK:`. Code reads them with `re.search`.
- **The budget:** with a limit of 30 calls, 28 are left for groups, so a group needs at least `math.ceil(200 / 28)` = 8 statements
- **Splitting a list:** `[items[start:start + size] for start in range(0, len(items), size)]`
- **The scorer:** `score_duplicates(found_ids)` returns how many planted duplicates were found and the false alarms. It accepts either charge of a pair.
- **The model:** every call of this exercise goes to the stronger model, with `model=LONG_JOB_MODEL`, at reasoning effort `low`. The small model of Exercises 1 to 4 cannot do this job: in our tests it reported hundreds of charges that are not duplicates, also on a group of ten statements.
- **Honest note:** this exercise does not show a gain in accuracy, and it is not meant to. In our four runs the single call found 56, 56, 55 and 56 of the 56 duplicates, with no false alarm. It needed about two minutes and wrote 30,000 to 36,000 tokens, nearly all of them thinking. The planner found all 56 with no false alarm, with 27 calls, in about half a minute and for about the same cost. The differences you can measure here are the waiting time, the cost and the number of calls. Topic 5 lists the situations where a plan pays; this file is not one where a single call fails. The plan itself is written anew in every run, so its wording and its group size can differ from ours. What does not vary: the budget your code enforces, and the extra calls on the simple question.
- **Honest note about cost:** this model's price is five times higher for a request above 100,000 tokens in. The whole file is about 86,500 tokens, below that. The cell prints the tokens of the single call so you can see how close it is.
- **Difficulty:** 🔴 Level 3, Independent

---

### 🚀 Level Up (optional)

1. Set `call_limit=12` for the statements and run it again. The groups get larger. Does the planner still find every duplicate, and how do its cost and its waiting time change?
2. Replace the final combining call by two lines of code that collect the `DUPLICATE:` lines of all groups. Compare the result and the cost. When does combining need a model at all?

In [ ]:
# ==========================================================================
# 🧩 Exercise 5: Have the Model Plan a Long Job and Solve It in Parts
# ==========================================================================

# Make sure the Setup cell and Exercise 1 ran first (Exercise 1 loads the fee table, used for the simple question)
assert "call_model" in globals(), "Run the Setup cell at the top of the notebook first."
assert "FEE_TABLE" in globals(), "Run Exercise 1 first: the simple question at the end uses its fee table."

# ==========================================================================
# Step 1: Load the statements (upload the file if it is not here yet)
# ==========================================================================
STATEMENTS_FILE = "bank_statements.json"
if not file_is_ready(STATEMENTS_FILE):
    print(f"📁 Please upload '{STATEMENTS_FILE}':")
    files.upload()
assert file_is_ready(STATEMENTS_FILE), f"'{STATEMENTS_FILE}' is missing or does not open. Upload it again (the exact name matters)."
print(f"📋 {STATEMENTS_FILE}: {describe_file(STATEMENTS_FILE)}")

with open(STATEMENTS_FILE) as f:
    statement_data = json.load(f)
STATEMENTS = statement_data["statements"]             # 200 statements, each with its list of transactions
DUPLICATE_RULE = statement_data["duplicate_rule"]     # the rule that makes a charge a duplicate, in words
PLANTED = statement_data["planted_duplicates"]        # 56 entries: statement_id, first_id, duplicate_id


# ==========================================================================
# Step 2: Turn statements into the text the model reads, and look at one
# ==========================================================================
# One header line per statement, then one line per charge: id, date, merchant, amount.
def statement_as_text(statement):
    """One statement as plain text."""
    lines = [f"Statement {statement['statement_id']} ({statement['period_start']} to {statement['period_end']})"]
    for charge in statement["transactions"]:
        lines.append(f"{charge['id']}  {charge['date']}  {charge['merchant']}  {charge['amount']:.2f}")
    return "\n".join(lines)


ALL_STATEMENT_TEXTS = [statement_as_text(statement) for statement in STATEMENTS]
charge_count = sum(len(statement["transactions"]) for statement in STATEMENTS)

print()
print("=" * 70)
print("📊 THE JOB")
print("=" * 70)
print(f"{len(STATEMENTS)} statements · {charge_count:,} charges · {len(PLANTED)} planted duplicates")
print("\nThe rule: " + textwrap.fill(DUPLICATE_RULE, width=100))
# Show a statement that has a planted duplicate, so you can see what one looks like
example_entry = PLANTED[0]
example_statement = next(s for s in STATEMENTS if s["statement_id"] == example_entry["statement_id"])
print("\n" + statement_as_text(example_statement))
print(f"\nIn this statement, {example_entry['duplicate_id']} is a duplicate of {example_entry['first_id']}.")


# ==========================================================================
# Step 3: The scorer (provided): how many planted duplicates were found, and how many false alarms?
# ==========================================================================
# A planted duplicate counts as found when the model reports either charge of the pair.
# A false alarm is a reported id that belongs to no planted pair.
def read_duplicates(text):
    """Return the set of charge ids on lines of the form 'DUPLICATE: T0042' in the text."""
    return set(re.findall(r"DUPLICATE:\s*[*_`]*\s*(T\d{4})", text))


def score_duplicates(found_ids):
    """Return (planted duplicates found, false alarms) for a set of reported charge ids."""
    found = sum(1 for entry in PLANTED if entry["duplicate_id"] in found_ids or entry["first_id"] in found_ids)
    ids_in_pairs = {entry["duplicate_id"] for entry in PLANTED} | {entry["first_id"] for entry in PLANTED}   # | joins two sets
    false_alarms = len(found_ids - ids_in_pairs)   # set minus set: the reported ids that are in no planted pair
    return found, false_alarms


# The job in words. The single call and the planner both get exactly this text.
STATEMENT_JOB = (
    "Find the duplicate charges in a set of bank statements. " + DUPLICATE_RULE + " "
    "The final answer must list every duplicate charge on its own line in the form DUPLICATE: <id of the second "
    "charge>, for example DUPLICATE: T0042, and nothing else. If there is no duplicate, the final answer is NONE."
)

# ==========================================================================
# Step 4: The base version: the whole file in one call
# ==========================================================================
# One call reads all 200 statements. max_tokens is much higher here: the model thinks its way through every
# statement, and thinking counts as written tokens (30,000 to 36,000 in our runs). A reply that is cut off in
# the middle would count as "found nothing", so we leave a lot of room.
whole_file_prompt = f"{STATEMENT_JOB}\n\n<statements>\n" + "\n\n".join(ALL_STATEMENT_TEXTS) + "\n</statements>"
whole_file_call = call_model(whole_file_prompt, max_tokens=60000, model=LONG_JOB_MODEL)
whole_file_ids = read_duplicates(whole_file_call["text"])
whole_file_found, whole_file_false_alarms = score_duplicates(whole_file_ids)

print()
print("=" * 70)
print("📊 THE WHOLE FILE IN ONE CALL")
print("=" * 70)
print(f"Tokens in             : {whole_file_call['tokens_in']:,} (the higher price starts at 100,000)")
print(f"Duplicates found      : {whole_file_found} of {len(PLANTED)}")
print(f"False alarms          : {whole_file_false_alarms}")
print(f"Calls, cost, seconds  : 1 call · ${whole_file_call['cost']:.4f} · {whole_file_call['seconds']:.1f} s")
# Show how the reply starts and ends, so you can see what the model really wrote
# tokens out include the thinking, which is not part of the reply text
print(f"The call wrote {whole_file_call['tokens_out']:,} tokens, thinking included (finish reason "
      f"'{whole_file_call['finish_reason']}'). Its reply starts with:")
print(textwrap.indent("\n".join(whole_file_call["text"].split("\n")[:5])[:400], "   "))   # its first 5 lines
print("and ends with:")
print(textwrap.indent("\n".join(whole_file_call["text"].strip().split("\n")[-5:]), "   "))   # the last five lines

# ==========================================================================
# Step 5: The planner prompt
# ==========================================================================
# The planner gets a description of the job, the number of items, one example item and the call budget.
# It does NOT get the data. Its output is three lines that our code can read.
PLANNER_PROMPT = """You plan a job for a team of assistants. Each assistant is one call to an AI model. \
The job is done in parts: the items are split into groups of equal size, one call works on each group, \
and one final call receives the findings of all groups and writes the final answer.

<job>
{job}
</job>

Number of items: {item_count}

<example_item>
{example_item}
</example_item>

Budget: at most {call_limit} calls in total. This planning call and the final call count, so at most \
{group_calls} calls are left for the groups.

Write the plan in exactly these three lines and nothing else:
GROUP_SIZE: <how many items one call gets, a whole number>
GROUP_TASK: <the instruction for a call that works on one group, with the exact form of its findings>
COMBINE_TASK: <the instruction for the final call that receives all findings>"""


def read_plan_line(plan_text, label):
    """Return the text after 'LABEL:' in the plan (up to the next label or the end), or '' if it is missing."""
    # (?s) lets "." match line breaks too, because an instruction can run over several lines
    match = re.search(rf"(?s){label}:\s*(.*?)(?=\n(?:GROUP_SIZE|GROUP_TASK|COMBINE_TASK):|\Z)", plan_text)
    return match.group(1).strip() if match else ""


# ==========================================================================
# Step 6: Plan, solve each group, combine
# ==========================================================================
def plan_and_solve(job, items, call_limit=30):
    """Have the model plan a job on a list of text items, run one call per group, and combine the findings.

    Returns a dictionary: the plan, the group size used, the groups, every call, the final answer,
    and the waiting time (measured, with the group calls running at the same time).
    """
    started = time.perf_counter()
    # TODO 1: How many calls are left for the groups? One call is the plan, one is the final combining call.
    group_calls_allowed = ___  # <-- Fill in the blank

    # ----- Stage 1: the plan (one call) -----
    plan_call = call_model(PLANNER_PROMPT.format(
        job=job, item_count=len(items), example_item=items[0], call_limit=call_limit, group_calls=group_calls_allowed,
    ), max_tokens=6000, model=LONG_JOB_MODEL)
    plan_text = plan_call["text"]
    size_text = re.search(r"\d+", read_plan_line(plan_text, "GROUP_SIZE"))   # the first whole number on that line
    planned_size = int(size_text.group()) if size_text else 0
    group_task = read_plan_line(plan_text, "GROUP_TASK") or "Do the job for the items below and list your findings."
    combine_task = read_plan_line(plan_text, "COMBINE_TASK") or "Combine the findings into the final answer."

    # ----- The budget, enforced by code -----
    # With N calls for groups, a group must hold at least ceil(items / N) items. ceil() rounds up.
    # If the plan asks for smaller groups (or gives no number), we use the smallest size the budget allows.
    # TODO 2: The smallest group size the budget allows: the number of items divided by the calls left
    # for groups, rounded up. Hint: math.ceil(...)
    smallest_allowed = ___  # <-- Fill in the blank
    # TODO 3: Use the plan's size, but never less than the smallest size the budget allows
    group_size = min(max(___, ___), len(items))  # <-- Fill in the blanks

    # ----- Stage 2: split the items into groups and run one call per group, at the same time -----
    # TODO 4: Cut the list of items into groups of group_size (the last group may be smaller).
    # Hint: [items[start:start + size] for start in range(0, len(items), size)]
    groups = ___  # <-- Fill in the blank

    def solve_group(group):
        prompt = (f"<job>\n{job}\n</job>\n\n<your_task>\n{group_task}\n</your_task>\n\n"
                  "<items>\n" + "\n\n".join(group) + "\n</items>")
        return call_model(prompt, max_tokens=12000, model=LONG_JOB_MODEL)

    # TODO 5: Run solve_group on every group, several at the same time. Hint: run_in_parallel(function, items)
    group_calls = ___  # <-- Fill in the blank

    # ----- Stage 3: one final call combines the findings -----
    findings = "\n\n".join(f"Findings of call {number}:\n{call['text']}"
                           for number, call in enumerate(group_calls, start=1))
    combine_call = call_model(
        f"<job>\n{job}\n</job>\n\n<your_task>\n{combine_task}\n</your_task>\n\n<findings>\n{findings}\n</findings>",
        max_tokens=16000, model=LONG_JOB_MODEL)

    # TODO 6: Every call this job made, in order: the plan call, the group calls (a list), the combine call
    calls = ___  # <-- Fill in the blank
    return {
        "plan": plan_text, "planned_size": planned_size, "group_size": group_size, "groups": groups,
        "calls": calls, "final": combine_call["text"],
        "cost": sum(call["cost"] for call in calls),
        "seconds_measured": time.perf_counter() - started,             # groups ran at the same time
        "seconds_in_a_row": sum(call["seconds"] for call in calls),    # if every call waited for the one before
    }


# ==========================================================================
# Step 7: Run the planner on the 200 statements
# ==========================================================================
CALL_LIMIT = 30
# TODO 7: Run the planner on the long job: the job in words is STATEMENT_JOB, the items are ALL_STATEMENT_TEXTS
planned = plan_and_solve(___, ___, call_limit=CALL_LIMIT)  # <-- Fill in the blanks
planned_ids = read_duplicates(planned["final"])
planned_found, planned_false_alarms = score_duplicates(planned_ids)

print()
print("=" * 70)
print("📊 THE PLAN THE MODEL WROTE")
print("=" * 70)
print(planned["plan"])
if planned["group_size"] != planned["planned_size"]:
    print(f"\n⚠️ The plan asked for groups of {planned['planned_size']}. The budget of {CALL_LIMIT} calls allows "
          f"groups of {planned['group_size']} or more, so the code used {planned['group_size']}.")
print(f"\nGroups: {len(planned['groups'])} groups of up to {planned['group_size']} statements · "
      f"calls used: {len(planned['calls'])} of {CALL_LIMIT} allowed")

print()
print("=" * 70)
print("📊 ONE CALL AGAINST THE PLANNER, ON 200 STATEMENTS")
print("=" * 70)
print(f"{'':<40}{'one call':>12}{'planner':>12}")
print(f"{'Duplicates found (of ' + str(len(PLANTED)) + ')':<40}{whole_file_found:>12}{planned_found:>12}")
print(f"{'False alarms':<40}{whole_file_false_alarms:>12}{planned_false_alarms:>12}")
print(f"{'Calls':<40}{1:>12}{len(planned['calls']):>12}")
print(f"{'Cost':<40}{'$' + format(whole_file_call['cost'], '.4f'):>12}{'$' + format(planned['cost'], '.4f'):>12}")
print(f"{'Seconds, group calls at the same time':<40}{whole_file_call['seconds']:>12.1f}{planned['seconds_measured']:>12.1f}")
print(f"{'Seconds, every call one after another':<40}{whole_file_call['seconds']:>12.1f}{planned['seconds_in_a_row']:>12.1f}")

record_result(
    task="200 statements, find duplicates", technique="single call",
    result=f"{whole_file_found} of {len(PLANTED)} found, {whole_file_false_alarms} false alarms",
    accuracy=whole_file_found / len(PLANTED), calls=1, cost=whole_file_call["cost"], seconds=whole_file_call["seconds"],
)
record_result(
    task="200 statements, find duplicates", technique="divide-and-conquer planning",
    result=f"{planned_found} of {len(PLANTED)} found, {planned_false_alarms} false alarms",
    accuracy=planned_found / len(PLANTED), calls=len(planned["calls"]), cost=planned["cost"],
    seconds=planned["seconds_in_a_row"], seconds_parallel=planned["seconds_measured"],
)

# ==========================================================================
# Step 8: The same planner on one simple question
# ==========================================================================
# One customer question with a one-line answer in the fee table. First a single call, then the same
# plan_and_solve with the question as the only item. The answer is checked with a pattern that finds "$4".
SIMPLE_QUESTION = "What does a paper statement sent by post cost per month?"
SIMPLE_ANSWER_PATTERN = r"\$\s?4(?:\.00)?(?!\d|,\d|\.\d)"   # "$4" or "$4.00", but not "$40" or "$4.50"
SIMPLE_JOB = (
    "Answer one customer question with the help of the bank's fee table. The final answer must be one or two "
    "sentences written to the customer, with the exact amount.\n\nThe fee table:\n" + FEE_TABLE
)

simple_single = call_model(f"{SIMPLE_JOB}\n\n<customer_question>\n{SIMPLE_QUESTION}\n</customer_question>",
                           max_tokens=ANSWER_MAX_TOKENS, model=LONG_JOB_MODEL)
# TODO 8: Run the same planner on the simple job: SIMPLE_JOB, with a list that holds one item, the question
simple_planned = plan_and_solve(___, ___, call_limit=CALL_LIMIT)  # <-- Fill in the blanks
simple_single_right = bool(re.search(SIMPLE_ANSWER_PATTERN, simple_single["text"]))
simple_planned_right = bool(re.search(SIMPLE_ANSWER_PATTERN, simple_planned["final"]))

print()
print("=" * 70)
print("📊 ONE CALL AGAINST THE PLANNER, ON ONE SIMPLE QUESTION")
print("=" * 70)
print(f"Question: {SIMPLE_QUESTION}")
print("\nSingle call: " + textwrap.fill(simple_single["text"], width=100))
print("\nPlanner, final answer: " + textwrap.fill(simple_planned["final"], width=100))
print(f"\n{'':<34}{'one call':>12}{'planner':>12}{'planner / one call':>20}")
print(f"{'Right answer ($4)':<34}{'yes' if simple_single_right else 'no':>12}{'yes' if simple_planned_right else 'no':>12}")
print(f"{'Calls':<34}{1:>12}{len(simple_planned['calls']):>12}{times_as_much(len(simple_planned['calls']), 1):>20}")
print(f"{'Cost':<34}{'$' + format(simple_single['cost'], '.5f'):>12}{'$' + format(simple_planned['cost'], '.5f'):>12}"
      f"{times_as_much(simple_planned['cost'], simple_single['cost']):>20}")
print(f"{'Seconds':<34}{simple_single['seconds']:>12.1f}{simple_planned['seconds_measured']:>12.1f}"
      f"{times_as_much(simple_planned['seconds_measured'], simple_single['seconds']):>20}")

record_result(
    task="1 simple question", technique="single call",
    result="right" if simple_single_right else "wrong", accuracy=float(simple_single_right),
    calls=1, cost=simple_single["cost"], seconds=simple_single["seconds"],
)
record_result(
    task="1 simple question", technique="divide-and-conquer planning",
    result="right" if simple_planned_right else "wrong", accuracy=float(simple_planned_right),
    calls=len(simple_planned["calls"]), cost=simple_planned["cost"],
    seconds=simple_planned["seconds_in_a_row"], seconds_parallel=simple_planned["seconds_measured"],
)
print()
show_bill([whole_file_call] + planned["calls"] + [simple_single] + simple_planned["calls"], "Exercise 5")

# ==========================================================================
# Step 9: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
grouped_items = [item for group in planned["groups"] for item in group]   # all groups, flattened into one list
print("✅ The groups of the plan cover all 200 statements, each exactly once" if grouped_items == ALL_STATEMENT_TEXTS
      else f"❌ The groups hold {len(grouped_items)} statements in all; expected the 200 statements in order, each once")
print(f"✅ The planner run used {len(planned['calls'])} calls, within the limit of {CALL_LIMIT}: 1 plan, "
      f"{len(planned['groups'])} groups, 1 combine"
      if len(planned["calls"]) <= CALL_LIMIT and len(planned["calls"]) == len(planned["groups"]) + 2
      else f"❌ The planner run used {len(planned['calls'])} calls for {len(planned['groups'])} groups; "
           f"the limit is {CALL_LIMIT}: check the budget in Step 6")
print("✅ On the simple question, the planner used more calls than the single call"
      if len(simple_planned["calls"]) > 1 else "❌ The planner should use at least a plan call, a group call and a combine call")
print("✅ The four rows of this exercise are stored in RESULTS" if sum(
      row["task"] in ("200 statements, find duplicates", "1 simple question") for row in RESULTS) == 4
      else "❌ A row of this exercise is missing in RESULTS")
if whole_file_call["finish_reason"] == "length":
    print("🤔 The single call's reply was cut off at max_tokens before it was finished, so its count is too low. "
          "Raise max_tokens in Step 4 and run the cell again to get a fair comparison.")
if whole_file_call["error"]:
    print(f"🤔 The single call on the whole file failed ({whole_file_call['error']}), so it found nothing. "
          "Run the cell again to get a fair comparison.")
if not planned_ids and planned_found == 0:
    print("🤔 The planner's final answer had no line of the form 'DUPLICATE: T0042'. Read the plan and the final "
          "answer above: the plan's COMBINE_TASK probably asked for another form. Run the cell again.")
elif planned_found > whole_file_found:
    print(f"🤔 On the long job the planner found {planned_found} of {len(PLANTED)} duplicates, the single call "
          f"{whole_file_found}. In our runs the single call found 55 or 56 of 56, so a duplicate or two missed "
          "in one run is not a gain you can rely on.")
elif planned_found == whole_file_found:
    print(f"🤔 On the long job both ways found {planned_found} of {len(PLANTED)} duplicates. The model with room to "
          "think works through the statements one by one in one call. The plan did not add accuracy here.")
else:
    print(f"🤔 On the long job the single call found {whole_file_found} of {len(PLANTED)} duplicates and the planner "
          f"{planned_found}. Read the plan and the findings of the groups: every extra call is one more place to slip.")
print(f"🤔 What the plan changed in your run: the wait was {planned['seconds_measured']:.0f} s against "
      f"{whole_file_call['seconds']:.0f} s for the single call (the group calls ran at the same time), "
      f"for {len(planned['calls'])} calls and {times_as_much(planned['cost'], whole_file_call['cost'])} the cost.")
if simple_planned["cost"] > simple_single["cost"]:
    print(f"🤔 On the simple question the planner cost {times_as_much(simple_planned['cost'], simple_single['cost'])} "
          f"the single call and took {times_as_much(simple_planned['seconds_measured'], simple_single['seconds'])} "
          "the time, for the same one-line answer. There was nothing to divide.")
else:
    print("🤔 On the simple question the planner did not cost more than the single call in your run, which is "
          "unusual (the single call probably thought for a long time). Look at the number of calls: it is still higher.")
print("🧠 A plan splits a long job into parts that run at the same time, can be read one by one, and stay within a budget. On one question it only adds calls.")

# ⚖️ Topic 6: Choosing a Technique From Accuracy, Cost and Response Time

You now have four techniques, and none of them is "better" in general. Each one buys something and charges for it:

| Technique | What your code does | What it can buy | What it always costs |
|---|---|---|---|
| **Self-consistency** | asks the same question several times and keeps the most common answer | fewer random slips, and a sign of how sure the answer is | as many times the calls as there are votes, and up to as many times the money |
| **Prompt chaining** | splits one job into calls with one small task each | steps you can read and check one by one, and steps that can use different inputs or tools | several calls that must wait for each other |
| **Reflection** | adds a call that reviews the first answer | mistakes caught, if the reviewer has something to check against | a second call, after the first |
| **Divide-and-conquer planning** | has the model plan, runs one call per part, combines | a long job done part by part | a plan call, a combine call, and many calls in between |

Which one to use depends on the situation the answer is needed in. Two situations cover most products:

- **A chat turn.** A person has asked something and is waiting. The limit that matters is the **response time**: after a couple of seconds people start to wonder whether anything is happening. Cost per answer matters too, because there are many turns. A technique whose calls must run one after another multiplies the wait.
- **A long job.** A program works through thousands of items overnight, and nobody is waiting for any single answer. Response time hardly matters. What matters is **accuracy** (nobody will check 10,000 results by hand) and the **total cost** of the whole job.

So the same technique can be the wrong choice in one situation and the right one in the other. A vote of five that makes a customer wait is hard to defend in a chat. The same vote on an overnight job costs a few extra dollars and nobody notices the time.

This is why you measured three numbers for everything. To choose, you put the rows next to each other and read across: what did the accuracy gain, how many times the cost, how many times the wait.

---

### 🔍 What You'll Learn

- Put several techniques into one table with the same columns
- Express each technique's cost and waiting time as a multiple of the single call
- Choose a technique for a situation from its limits: a response time for a chat, accuracy and total cost for a long job

### 🧠 Why This Matters

- Teams add "a quick review step" or "a little voting" because it sounds careful, and find out later that chat answers became three times slower
- "The cheapest thing that is good enough for this situation" is a decision you can defend with a table. "The most advanced technique" is not

## 🧪 Exercise 6: Compare the Four Techniques and Choose One for Two Situations

Every exercise stored its rows in `RESULTS`. Now you turn them into **one table**: a row for the single call and for each technique, with the accuracy, the number of calls, the cost and the response time of one answer. Each technique was tried on the job that suits it, so the table is grouped by job, and each technique stands next to the single call on the same job. The chain of Exercise 3 was run by itself, so its row has no single call next to it and no multiples. You add two columns that make the rows easy to compare: the cost and the waiting time as a multiple of the single call.

Then you use the table to answer two questions, in one sentence each:

1. Which technique, if any, would you use in a **chat** where a customer is waiting for an answer within **two seconds**?
2. Which would you use for a job that runs **overnight on 10,000 statements**?

The notebook helps with two small calculations: which rows stay within two seconds, and what each way of checking statements would cost for 10,000 of them, how long it would take, and whether 10,000 statements fit into one call.

### 🎯 What You'll Do

1. Build the table from `RESULTS`.
2. Add the two "times the single call" columns.
3. Find the rows that answer within two seconds, and work out the cost of 10,000 statements.
4. Write your two choices, each with a one-sentence reason.

👉 By the end, you'll have the table and two choices you can defend with its numbers.

---

### ✅ You Succeed When

- The table has all 9 rows, and every job except the complaints has its single-call row
- The cost and time multiples of every single-call row are exactly 1
- Each of your two choices names a technique from the table (or "none") and has a one-sentence reason

---

### 🛠 Technical Info

- **A table from a list of dictionaries:** `pd.DataFrame(RESULTS)`; a column is `table["cost"]`
- **The single call of the same job:** look it up by the row's `task`
- **The two multiples:** `cost x single call` and `time x single call` divide a row by the single call of the same job. The time multiple uses the seconds with the calls one after another.
- **Honest note:** the three jobs are different, so compare a technique only with the single call **on the same job**, never across jobs. "Seconds" were measured while several calls ran at the same time in this notebook, which is good enough to compare rows, and a real service can be faster or slower. The single call itself can already take about two seconds.
- **Honest note about your choices:** there is no single right answer. The check only looks at whether you named a technique and gave a reason. The 🤔 lines compare your choice with your own numbers.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Add a column "cost of one more right answer": the extra cost of a technique divided by the extra right answers it gave on the 15 calculations.
2. Combine two techniques: review with the fee table only the questions where the vote of five was split. Estimate its cost per answer from the numbers you already have.

In [ ]:
# ==========================================================================
# ⚖️ Exercise 6: Compare the Four Techniques and Choose One for Two Situations
# ==========================================================================

# Make sure the earlier exercises ran first (each one stores its rows in RESULTS)
assert "RESULTS" in globals(), "Run the Setup cell at the top of the notebook first."
assert len(RESULTS) == 9, f"RESULTS has {len(RESULTS)} of 9 rows. Run Exercises 1 to 5 first, from top to bottom."

# ==========================================================================
# Step 1: Build the table from RESULTS
# ==========================================================================
# One row per job and technique, in a fixed order: the jobs as they appeared, the single call first in each job.
TASK_ORDER = ["15 fee calculations", "20 complaints with 3 issues", "200 statements, find duplicates", "1 simple question"]
rows = sorted(RESULTS, key=lambda row: (TASK_ORDER.index(row["task"]), row["technique"] != "single call"))
# TODO 1: Turn the list of rows into a table. Hint: pd.DataFrame(a list of dictionaries)
table = ___  # <-- Fill in the blank

# ==========================================================================
# Step 2: Add cost and time as a multiple of the single call on the same job
# ==========================================================================
# For each row we look up the single-call row of the same job and divide by it.
# The chain on the complaints has no single-call row, so it gets no multiple (shown as a dash).
# TODO 2: Keep only the rows of the base version. Hint: its technique is called "single call"
single_rows = {row["task"]: row for row in rows if row["technique"] == ___}  # <-- Fill in the blank (a string)


def multiple(row, key):
    """row[key] divided by the same value of the single call on the same job (None if that job has none)."""
    base = single_rows.get(row["task"])   # .get() returns None when the job has no single-call row
    if base is None:
        return None
    return row[key] / base[key] if base[key] else 0


table["cost x single call"] = [multiple(row, "cost") for row in rows]
table["time x single call"] = [multiple(row, "seconds") for row in rows]

print("=" * 70)
print("📊 THE COMPARISON TABLE (calls, cost and seconds are for ONE answer or one whole job)")
print("=" * 70)
shown_table = table.rename(columns={
    "task": "job", "result": "result", "calls": "calls", "cost": "cost (USD)",
    "seconds": "seconds, calls in a row", "seconds_parallel": "seconds, parallel where possible",
}).drop(columns=["accuracy"])
display(shown_table.round({"cost (USD)": 5, "seconds, calls in a row": 1, "seconds, parallel where possible": 1,
                           "cost x single call": 1, "time x single call": 1}).fillna("-"))   # an empty cell is shown as a dash

# ==========================================================================
# Step 3: Two small calculations for the two situations
# ==========================================================================
# Situation 1, the chat: which rows answer within two seconds, even with parallel calls where possible?
CHAT_LIMIT_SECONDS = 2.0
fast_enough = [row for row in rows if row["seconds_parallel"] <= CHAT_LIMIT_SECONDS]
print()
print("=" * 70)
print("📊 SITUATION 1: A CHAT, ANSWER WITHIN 2 SECONDS")
print("=" * 70)
for row in rows:
    if row["task"] != "200 statements, find duplicates":   # a chat turn is one question, not a whole file
        verdict = "within 2 s" if row in fast_enough else "too slow"
        print(f"{row['technique']:<40}{row['task']:<30}{row['seconds_parallel']:>6.1f} s   {verdict}")
if not any(row["technique"] == "single call" for row in fast_enough):
    print("\nIn your run even the single call needed more than 2 seconds. No technique that adds calls can fix "
          "that: you would lower the reasoning effort or choose a faster model first.")

# Situation 2, the overnight job: what would 10,000 statements cost and how long would they take?
print()
print("=" * 70)
print("📊 SITUATION 2: AN OVERNIGHT JOB ON 10,000 STATEMENTS")
print("=" * 70)
JOB_SIZE = 10_000
for row in rows:
    if row["task"] == "200 statements, find duplicates":
        # The measured job had 200 statements, so 10,000 statements are 50 times that job
        # TODO 3: Scale the measured cost from 200 statements to JOB_SIZE statements
        cost_for_job = ___  # <-- Fill in the blank
        calls_for_job = row["calls"] * JOB_SIZE / 200
        minutes_for_job = row["seconds_parallel"] * JOB_SIZE / 200 / 60   # one file of 200 after the other
        print(f"{row['technique']:<30} found {row['accuracy']:.0%} of the duplicates · for {JOB_SIZE:,} statements: "
              f"about {calls_for_job:,.0f} calls, ${cost_for_job:,.2f} and {minutes_for_job:,.0f} minutes")
# Do 10,000 statements fit into ONE call? The 200 statements of Exercise 5 took whole_file_call["tokens_in"] tokens.
tokens_for_job = whole_file_call["tokens_in"] * JOB_SIZE / 200
print(f"\n{JOB_SIZE:,} statements are about {tokens_for_job / 1_000_000:.1f} million tokens. The model can read "
      f"{LONG_JOB_CONTEXT_TOKENS / 1_000_000:.0f} million in one call, so the job "
      f"{'fits into one call' if tokens_for_job <= LONG_JOB_CONTEXT_TOKENS else 'does not fit into one call: it has to be split'}.")

# ==========================================================================
# Step 4: Your two choices, one sentence each
# ==========================================================================
# Name a technique exactly as it appears in the table, or "none" if you would use a single call.
TECHNIQUES = sorted({row["technique"] for row in rows} - {"single call"})   # the technique names in the table

# TODO 4: Your choice for the chat (a technique name from the table, or "none") and your reason, in one
# sentence that uses a number from your table
CHAT_CHOICE = ___  # <-- Fill in the blank (a string)
CHAT_REASON = ___  # <-- Fill in the blank (a string, one sentence)
# TODO 5: Your choice for the overnight job on 10,000 statements, and your reason in one sentence
OVERNIGHT_CHOICE = ___  # <-- Fill in the blank (a string)
OVERNIGHT_REASON = ___  # <-- Fill in the blank (a string, one sentence)

print()
print("=" * 70)
print("📝 YOUR TWO CHOICES")
print("=" * 70)
print(f"Chat, answer within 2 seconds  : {CHAT_CHOICE}")
print("   " + textwrap.fill(CHAT_REASON, width=100, subsequent_indent="   "))
print(f"Overnight, 10,000 statements   : {OVERNIGHT_CHOICE}")
print("   " + textwrap.fill(OVERNIGHT_REASON, width=100, subsequent_indent="   "))

# ==========================================================================
# Step 5: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ The table has all 9 rows, and every job except the complaints has its single-call row"
      if len(table) == 9 and set(single_rows) == set(TASK_ORDER) - {"20 complaints with 3 issues"}
      else f"❌ Expected 9 rows and a single-call row for 3 of the 4 jobs, got {len(table)} rows")
single_multiples = table[table["technique"] == "single call"]   # only the single-call rows of the table
print("✅ The cost and time multiples of every single-call row are exactly 1"
      if all(abs(value - 1) < 1e-9 or value == 0 for value in single_multiples["cost x single call"])
      and all(abs(value - 1) < 1e-9 or value == 0 for value in single_multiples["time x single call"])
      else "❌ A single-call row does not have the multiple 1: check the division in Step 2")


def one_sentence(text):
    """True for a text of at least 5 words with at most one sentence end (. ! ?) followed by more text."""
    return len(text.split()) >= 5 and len(re.findall(r"[.!?]\s+\S", text.strip())) == 0


choices_ok = (CHAT_CHOICE in TECHNIQUES + ["none"] and OVERNIGHT_CHOICE in TECHNIQUES + ["none"]
              and one_sentence(CHAT_REASON) and one_sentence(OVERNIGHT_REASON))
print("✅ Each of your two choices names a technique from the table (or 'none') and has a one-sentence reason"
      if choices_ok else "❌ A choice is not a technique name from the table (or 'none'), or a reason is not one "
                         f"sentence. The names: {', '.join(TECHNIQUES)}")
chat_rows = [row for row in rows if row["technique"] == CHAT_CHOICE]
if CHAT_CHOICE == "none":
    print("🤔 For the chat you chose no technique: a single call. Check it against your table: is the single call "
          "itself within 2 seconds in your run?")
elif chat_rows and min(row["seconds_parallel"] for row in chat_rows) > CHAT_LIMIT_SECONDS:
    print(f"🤔 For the chat you chose '{CHAT_CHOICE}', which took "
          f"{min(row['seconds_parallel'] for row in chat_rows):.1f} s per answer in your run, more than the 2 seconds "
          "the customer has. Is its gain in accuracy worth that wait?")
else:
    print(f"🤔 For the chat you chose '{CHAT_CHOICE}'. In your run it stayed within 2 seconds. Check what it gained "
          "in accuracy over the single call on the same job.")
overnight_rows = [row for row in rows if row["technique"] == OVERNIGHT_CHOICE]
if OVERNIGHT_CHOICE == "none":
    print("🤔 For the overnight job you chose a single call. Look at the token line of Situation 2: do 10,000 "
          "statements fit into one call of this model?")
elif overnight_rows:
    print(f"🤔 For the overnight job you chose '{OVERNIGHT_CHOICE}'. Time is not the limit there, so check three "
          "things for the whole job: the duplicates found, the cost, and whether the job fits into one call at all.")
print("🧠 Choose per situation: a chat is limited by the wait, a long job by accuracy, total cost and what fits into one call.")

# 🏁 Wrap-up: What You Did

| Exercise | What you did | Level |
|---|---|---|
| 1. Measure the Baseline With One Call | Measured accuracy, cost and response time of one call on 15 fee calculations | 🟢 |
| 2. Ask Each Question Five Times | Built a majority vote over five answers, and saw that split votes mark the hard questions | 🟢 |
| 3. Split the Reply Into a Chain of Three Calls | Built a chain (list the issues, find the policy, write the reply) and read its intermediate results | 🟡 |
| 4. Add a Second Call That Reviews the First Answer | Compared a bare review with a review against the fee table, counting changes in both directions | 🟡 |
| 5. Have the Model Plan a Long Job | Built a planner with a call budget for 200 statements, compared it with one long call as it is, then saw it add only cost on one simple question | 🔴 |
| 6. Compare the Four Techniques | Put every result in one table and chose a technique for a chat and for an overnight job | 🟡 |

### 🧾 The habit
Before you add calls: **measure the single call first.** Accuracy, cost, response time.

### 🔑 The sentences to keep
- *These techniques are code around the model: a loop and a vote, three calls in a row, a second call that reviews, a plan with a budget.*
- *Each one costs calls, money and waiting time. What it buys differs: more right answers from a vote or a review with something to check against, steps you can read and check from a chain, a long job done in parts from a plan. Whether that is a good trade depends on who is waiting for the answer.*